# Bölüm 18 – Autoencoders, GANs ve Diffusion Models

> **Türkçe Açıklamalı Notebook**  
> Orijinal kaynak: *Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow* (Aurélien Géron)  
> Bu notebook, bölüm 18'deki tüm örnek kodları ve alıştırmaları içermektedir.

---

### Bu Notebook'ta Ne Öğreneceğiz?

| Konu | Açıklama |
|------|----------|
| **Undercomplete Linear Autoencoder** | PCA benzeri boyut indirgeme |
| **Stacked Autoencoder** | Derin katmanlı encoder-decoder mimarisi |
| **Anomaly Detection** | Yeniden oluşturma kaybıyla anormallik tespiti |
| **Tied Weights Autoencoder** | Ağırlıkların paylaşılması |
| **Convolutional Autoencoder** | CNN tabanlı encoder-decoder |
| **Recurrent Autoencoder** | LSTM tabanlı encoder-decoder |
| **Denoising Autoencoder** | Gürültü giderme amacıyla eğitim |
| **Sparse Autoencoder** | Seyrek temsil öğrenme |
| **Variational Autoencoder (VAE)** | Sürekli latent space ile üretici model |
| **Discrete VAE** | Gumbel-Softmax ile kesikli latent space |
| **GAN** | Üretici Çekişmeli Ağlar |
| **Deep Convolutional GAN (DCGAN)** | CNN tabanlı GAN |
| **Diffusion Models (DDPM & DDIM)** | İleri/geri diffusion süreci |
| **Stable Diffusion (sd-turbo)** | Hazır text-to-image modeli kullanımı |

# 1. Kurulum (Setup)

Bu bölümde, notebook'un çalışabilmesi için gerekli olan ortam kontrolleri ve kütüphane kurulumları yapılmaktadır.

## 1.1 Python Sürüm Kontrolü

Bu proje **Python 3.10 veya üzerini** gerektirmektedir. `sys.version_info` ile mevcut Python sürümü kontrol edilir. Eğer sürüm 3.10'dan düşükse `assert` ifadesi bir `AssertionError` fırlatır ve notebook çalışmayı durdurur.

In [ ]:
import sys

# Python sürümünün 3.10 veya üzeri olduğunu kontrol ediyoruz.
# Eğer daha eski bir Python sürümü kullanılıyorsa AssertionError alırsınız.
assert sys.version_info >= (3, 10)

## 1.2 Çalışma Ortamı Tespiti

Notebook'un Google Colab'da mı yoksa Kaggle'da mı çalıştığını tespit ediyoruz. Bu bilgi, platform özelinde paket kurulumu veya uyarı mesajları için kullanılmaktadır.

- `IS_COLAB`: `google.colab` modülü `sys.modules` içindeyse Colab'dayız demektir.
- `IS_KAGGLE`: `kaggle_secrets` modülü mevcutsa Kaggle ortamındayız demektir.

In [ ]:
# Çalışma ortamını tespit ediyoruz.
# sys.modules, Python'ın o anda yüklenmiş olan tüm modüllerin kayıtlı olduğu sözlüktür.
IS_COLAB = "google.colab" in sys.modules   # Colab'da çalışıyorsak True
IS_KAGGLE = "kaggle_secrets" in sys.modules  # Kaggle'da çalışıyorsak True

## 1.3 Gerekli Kütüphanelerin Kurulumu

`torchmetrics` kütüphanesi, model performansını ölçmek için kullanılır (örneğin RMSE, accuracy). Google Colab bu kütüphaneyi varsayılan olarak içermez, bu yüzden `%pip install` ile kuruyoruz.

In [ ]:
# Eğer Google Colab'da çalışıyorsak, torchmetrics kütüphanesini kuruyoruz.
# torchmetrics: model metriklerini (RMSE, Accuracy vb.) hesaplamak için PyTorch uyumlu kütüphane
if IS_COLAB:
    %pip install -q torchmetrics

## 1.4 PyTorch Sürüm Kontrolü

Bu notebook **PyTorch 2.6.0 veya üzerini** gerektirir. `packaging.version.Version` sınıfı, sürüm karşılaştırmalarını güvenli biçimde yapmamızı sağlar.

In [ ]:
from packaging.version import Version  # Sürüm karşılaştırması için
import torch                            # Ana derin öğrenme kütüphanesi

# PyTorch sürümünün 2.6.0 veya üzeri olduğunu kontrol ediyoruz.
# Eğer daha eski bir sürüm kullanılıyorsa AssertionError alırsınız.
assert Version(torch.__version__) >= Version("2.6.0")

## 1.5 Donanım Hızlandırıcı (Device) Seçimi

Derin öğrenme modelleri CPU'da çok yavaş çalışır. Bu yüzden mevcut donanıma göre en iyi device'ı seçiyoruz:

- **`cuda`**: NVIDIA GPU — en hızlı seçenek
- **`mps`**: Apple Silicon (M1/M2/M3) GPU — macOS için
- **`cpu`**: Donanım hızlandırıcı yoksa fallback

In [ ]:
# Kullanılabilir donanım hızlandırıcıya göre device belirliyoruz.
# Sıralama: CUDA (NVIDIA GPU) > MPS (Apple Silicon) > CPU

if torch.cuda.is_available():
    device = "cuda"          # NVIDIA GPU kullanılabilir
elif torch.backends.mps.is_available():
    device = "mps"           # Apple Silicon GPU kullanılabilir
else:
    device = "cpu"           # Hiçbir GPU yoksa CPU kullan

device  # Seçilen device'ı yazdırıyoruz

## 1.6 Donanım Hızlandırıcı Yoksa Uyarı

Eğer GPU bulunamazsa ve model yalnızca CPU'da çalışıyorsa, kullanıcıyı uyarıyoruz ve Colab/Kaggle'da GPU nasıl etkinleştirilir onu açıklıyoruz.

In [ ]:
# Eğer GPU yoksa, kullanıcıya uyarı mesajı göster ve GPU nasıl etkinleştirilir anlat.
if device == "cpu":
    print("Neural nets can be very slow without a hardware accelerator.")
    if IS_COLAB:
        # Colab'da GPU etkinleştirme yolu:
        print("Go to Runtime > Change runtime and select a GPU hardware accelerator.")
    if IS_KAGGLE:
        # Kaggle'da GPU etkinleştirme yolu:
        print("Go to Settings > Accelerator and select GPU.")

## 1.7 Matplotlib Grafik Ayarları

Grafiklerin daha okunabilir görünmesi için `matplotlib.pyplot.rc()` fonksiyonuyla varsayılan yazı tipi boyutlarını ayarlıyoruz.

- `font size=14`: Genel yazı boyutu
- `axes labelsize/titlesize=14`: Eksen etiketleri ve grafik başlıkları
- `legend fontsize=14`: Lejand yazı boyutu
- `xtick/ytick labelsize=10`: Eksen ölçek etiketi boyutları

In [ ]:
import matplotlib.pyplot as plt

# Tüm grafiklerde kullanılacak varsayılan yazı tipi boyutlarını ayarlıyoruz.
# Bu, grafiklerimizin kitap ve sunum kalitesinde görünmesini sağlar.
plt.rc('font', size=14)             # Genel yazı boyutu
plt.rc('axes', labelsize=14, titlesize=14)  # Eksen etiketleri ve başlık boyutu
plt.rc('legend', fontsize=14)       # Lejand yazı boyutu
plt.rc('xtick', labelsize=10)       # X ekseni ölçek etiketi boyutu
plt.rc('ytick', labelsize=10)       # Y ekseni ölçek etiketi boyutu

## 1.8 Yardımcı Eğitim Fonksiyonları: `evaluate_tm()` ve `train()`

Bu iki fonksiyon, notebook boyunca tüm modelleri eğitmek ve değerlendirmek için kullanılacaktır.

### `evaluate_tm(model, data_loader, metric)`
Bir modeli **değerlendirme moduna** alarak validation/test seti üzerinde metrik hesaplar.

### `train(model, optimizer, loss_fn, metric, train_loader, valid_loader, n_epochs, ...)`
Bir modeli tam eğitim döngüsüyle eğitir. Önceki bölümlerdeki `train()` fonksiyonuna göre **iki önemli iyileştirme** içerir:

1. **Gradient Clipping** (`clip_grad_norm_`): Gradient'lerin normunu 1.0 ile sınırlayarak exploding gradient problemini önler.
2. **Named Tuple Desteği**: `VAE` ve `SparseAutoencoder` gibi modeller `(output, codings)` şeklinde named tuple döndürür. `y_pred.output` satırı bu durumu ele alır.


In [ ]:
import torchmetrics

def evaluate_tm(model, data_loader, metric):
    """
    Modeli evaluation moduna alır ve tüm veri seti üzerinde metrik hesaplar.
    
    Args:
        model: Değerlendirilecek PyTorch modeli
        data_loader: Değerlendirme için DataLoader
        metric: torchmetrics metrik nesnesi (örn. RMSE)
    
    Returns:
        Hesaplanan metrik değeri (scalar tensor)
    """
    model.eval()    # Dropout ve BatchNorm'u değerlendirme moduna al
    metric.reset()  # Önceki birikmiş metrik değerlerini sıfırla
    with torch.no_grad():  # Gradient hesaplamayı kapat (bellek tasarrufu)
        for X_batch, y_batch in data_loader:
            # Veriyi GPU/CPU'ya taşı
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            # Eğer model (output, codings) gibi bir tuple döndürüyorsa sadece output'u al
            if isinstance(y_pred, tuple):
                y_pred = y_pred.output
            metric.update(y_pred, y_batch)  # Metriği güncelle
    return metric.compute()  # Tüm batch'lerin ortalamasını hesapla


def train(model, optimizer, loss_fn, metric, train_loader, valid_loader,
          n_epochs, patience=2, factor=0.5, epoch_callback=None):
    """
    Bir PyTorch modelini verilen parametrelerle eğitir.
    
    Args:
        model: Eğitilecek PyTorch modeli
        optimizer: Optimizasyon algoritması (NAdam, Adam vb.)
        loss_fn: Kayıp fonksiyonu (MSELoss, CrossEntropyLoss vb.)
        metric: torchmetrics metrik nesnesi
        train_loader: Eğitim DataLoader'ı
        valid_loader: Doğrulama DataLoader'ı
        n_epochs: Eğitim epoch sayısı
        patience: ReduceLROnPlateau için bekleme epoch sayısı
        factor: Öğrenme hızı azaltma çarpanı
        epoch_callback: Her epoch başında çağrılacak opsiyonel fonksiyon
                        (örn. Discrete VAE'de temperature annealing için)
    
    Returns:
        history: Eğitim kaybı ve metriklerin kaydı (dict)
    """
    # ReduceLROnPlateau: validation metriği iyileşmediğinde learning rate'i düşürür
    # mode='max': metriğin artmasını istiyoruz (örn. accuracy)
    # patience=2: 2 epoch ilerleme olmazsa lr'yi düşür
    # factor=0.5: lr'yi yarıya indir
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", patience=patience, factor=factor)
    
    # Eğitim geçmişini saklamak için sözlük
    history = {"train_losses": [], "train_metrics": [], "valid_metrics": []}
    
    for epoch in range(n_epochs):
        total_loss = 0.0
        metric.reset()    # Epoch başında metriği sıfırla
        model.train()     # Modeli eğitim moduna al (Dropout ve BatchNorm aktif)
        
        # Opsiyonel epoch callback'i çağır (örn. DiscreteVAE'de temperature annealing)
        if epoch_callback is not None:
            epoch_callback(model, epoch)
        
        for index, (X_batch, y_batch) in enumerate(train_loader):
            # Veriyi doğru device'a taşı
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            
            # İleri geçiş (forward pass): tahmin üret
            y_pred = model(X_batch)
            
            # Kayıp hesapla
            loss = loss_fn(y_pred, y_batch)
            total_loss += loss.item()
            
            # Geri yayılım (backward pass): gradient hesapla
            loss.backward()
            
            # ─── Gradient Clipping ───────────────────────────────────────────
            # Tüm parametrelerin gradient normunu maksimum 1.0 ile sınırla.
            # Bu, exploding gradient problemini önler ve eğitimi stabilize eder.
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            # ────────────────────────────────────────────────────────────────
            
            # Parametreleri güncelle
            optimizer.step()
            optimizer.zero_grad()  # Gradient'leri sıfırla
            
            # Named tuple desteği: model (output, codings) döndürüyorsa sadece output al
            if isinstance(y_pred, tuple):
                y_pred = y_pred.output
            
            # Metriği batch sonucuyla güncelle
            metric.update(y_pred, y_batch)
            train_metric = metric.compute().item()
            
            # İlerlemeyi ekrana yazdır (aynı satırı güncelle)
            print(f"\rBatch {index + 1}/{len(train_loader)}", end="")
            print(f", loss={total_loss/(index+1):.4f}", end="")
            print(f", {train_metric=:.3f}", end="")
        
        # Epoch metriklerini kaydet
        history["train_losses"].append(total_loss / len(train_loader))
        history["train_metrics"].append(train_metric)
        
        # Validation metriğini hesapla
        val_metric = evaluate_tm(model, valid_loader, metric).item()
        history["valid_metrics"].append(val_metric)
        
        # Scheduler'ı validation metriğine göre güncelle
        scheduler.step(val_metric)
        
        # Epoch özetini yazdır
        print(f"\rEpoch {epoch + 1}/{n_epochs},                      "
              f"train loss: {history['train_losses'][-1]:.4f}, "
              f"train metric: {history['train_metrics'][-1]:.3}, "
              f"valid metric: {history['valid_metrics'][-1]:.3}")
    
    return history

# 2. Undercomplete Linear Autoencoder ile PCA

## Temel Kavramlar

**Autoencoder** (Otokodlayıcı), giriş verisini önce daha düşük boyutlu bir temsile (**encoding** / **latent space**) sıkıştıran, ardından bu temsili orijinal boyuta geri açan (**decoding**) bir sinir ağıdır.

```
Giriş (x)  →  [Encoder]  →  Coding (z)  →  [Decoder]  →  Çıkış (x̂)
```

**Undercomplete Autoencoder**: Coding boyutunun giriş boyutundan küçük olduğu durum.  
**Linear Autoencoder** aktivasyon fonksiyonu olmayan, sadece linear katmanlardan oluşur.

> **İlginç Gerçek**: Doğrusal aktivasyonlu undercomplete bir autoencoder, **PCA** (Principal Component Analysis) ile matematiksel olarak eşdeğerdir! Ağ, verinin en fazla varyansı açıklayan eksenleri öğrenir.

Bu örnekte:
- **Giriş**: 3 boyutlu (3D) veri
- **Coding**: 2 boyutlu (2D) — boyut indirgeme
- **Çıkış**: Tekrar 3 boyutlu


## 2.1 Basit Linear Autoencoder Oluşturma

Encoder 3→2, decoder 2→3 boyutuna geçiş yapan birer linear katmandan oluşuyor. `nn.Sequential` ile bunları ardışık bağlıyoruz.

In [ ]:
import torch
import torch.nn as nn

# Reproducibility için random seed sabitleme
torch.manual_seed(42)

# Encoder: 3 boyutlu girişi 2 boyutlu coding'e sıkıştırır
# nn.Linear(in_features, out_features) — aktivasyon yok (doğrusal)
encoder = nn.Linear(3, 2)

# Decoder: 2 boyutlu coding'i 3 boyutlu çıkışa genişletir
decoder = nn.Linear(2, 3)

# Encoder ve decoder'ı ardışık bağlayarak tam autoencoder oluştur
# Giriş → encoder → coding → decoder → çıkış
autoencoder = nn.Sequential(encoder, decoder).to(device)

## 2.2 3D Veri Seti Oluşturma

Bölüm 7'de kullanılan 3 boyutlu elips şeklindeki veri setini yeniden oluşturuyoruz. Veri noktaları:
1. Bir oval/elips üzerinde açısal dağılım ile üretiliyor
2. Rastgele gürültü ekleniyor
3. 3D uzayda döndürülüyor (rotation)
4. Biraz kaydırılıyor (shift)

In [ ]:
import numpy as np
from scipy.spatial.transform import Rotation  # 3D rotasyon için

def generate_data(m, seed=42):
    """
    3 boyutlu, elips şeklinde bir veri seti oluşturur.
    
    Args:
        m: Üretilecek veri noktası sayısı
        seed: Rastgelelik için seed değeri
    
    Returns:
        torch.Tensor: (m, 3) boyutunda float32 tensor
    """
    X = np.zeros((m, 3))  # m adet 3D nokta için boş dizi
    rng = np.random.default_rng(seed)
    
    # Açıları küp kök ile çarparak düzensiz (non-uniform) dağılım oluştur
    # Bu sayede noktalar elips üzerinde eşit dağılmaz, daha gerçekçi görünür
    angles = (rng.random(m) ** 3 + 0.5) * 2 * np.pi
    
    # X ve Y koordinatları: yatay eksende tam çember, dikeyde yarı (oval)
    X[:, 0], X[:, 1] = np.cos(angles), np.sin(angles) * 0.5
    
    # Tüm 3 boyuta Gaussian gürültü ekle (std=0.28)
    X += 0.28 * rng.standard_normal((m, 3))
    
    # Veriyi 3D uzayda döndür:
    # - X ekseni etrafında: pi/29 radyan
    # - Y ekseni etrafında: -pi/20 radyan  
    # - Z ekseni etrafında: pi/4 radyan (45 derece)
    X = Rotation.from_rotvec([np.pi / 29, -np.pi / 20, np.pi / 4]).apply(X)
    
    # Veriyi biraz kaydır (merkezi değiştir)
    X += [0.2, 0, 0.2]
    
    # NumPy float64'ten PyTorch float32 tensor'a dönüştür
    return torch.from_numpy(X.astype(np.float32))

## 2.3 Training ve Validation DataLoader Oluşturma

Autoencoder'lar **unsupervised** (gözetimsiz) öğrenme modelleridir: hedef çıktı, giriş ile aynıdır. Bu yüzden `TensorDataset(X, X)` şeklinde hem giriş hem hedef olarak aynı veriyi kullanıyoruz.

In [ ]:
from torch.utils.data import DataLoader, TensorDataset

# Eğitim verisi: 60 nokta
X_train = generate_data(60, seed=42)

# TensorDataset(X_train, X_train):
# Autoencoder'da hedef = giriş (yeniden oluşturma görevi)
# Yani model X'i öğrenip yine X üretmeye çalışır
train_set = TensorDataset(X_train, X_train)
train_loader = DataLoader(train_set, batch_size=32, shuffle=True)

In [ ]:
# Validation verisi: 500 nokta (daha büyük, değerlendirme için)
X_valid = generate_data(500, seed=43)
valid_set = TensorDataset(X_valid, X_valid)
valid_loader = DataLoader(valid_set, batch_size=32)

## 2.4 Modeli Eğitme

- **NAdam optimizer**: Adam'ın Nesterov momentumlu versiyonu, genellikle Adam'dan daha hızlı yakınsar
- **MSELoss**: Yeniden oluşturma kaybı — üretilen çıktı ile orijinal giriş arasındaki ortalama kare hata
- **RMSE metrik**: MSE'nin karekökü, daha yorumlanabilir

In [ ]:
import torchmetrics

torch.manual_seed(42)

# NAdam optimizer: Adam + Nesterov momentum kombinasyonu
# lr=0.2: Yüksek learning rate, basit doğrusal model için yeterli
optimizer = torch.optim.NAdam(autoencoder.parameters(), lr=0.2)

# MSE Loss: reconstruction loss olarak kullanıyoruz
# Model, giriş ile çıkışı olabildiğince benzer yapmaya çalışacak
mse = nn.MSELoss()

# RMSE metrik: squared=False → MSE'nin değil RMSE'nin hesaplanmasını sağlar
rmse = torchmetrics.MeanSquaredError(squared=False).to(device)

# Modeli 20 epoch eğit
history = train(autoencoder, optimizer, mse, rmse, train_loader, valid_loader,
                n_epochs=20)

## 2.5 Coding'leri Çıkarma

Eğitim tamamlandıktan sonra encoder'ı kullanarak eğitim verilerinin 2D latent space temsillerini (coding) elde ediyoruz.

In [ ]:
# Encoder'dan 2 boyutlu coding vektörlerini elde et
# X_train.to(device): tensor'u GPU/CPU'ya taşı
codings = encoder(X_train.to(device))
# codings.shape: [60, 2] — 60 nokta, her biri 2 boyutlu

## 2.6 Coding'leri Görselleştirme

Encoder'ın 3D veriyi 2D'ye nasıl sıkıştırdığını görmek için latent space'deki noktaları çiziyoruz. Bu grafik bize PCA benzeri ana bileşenleri gösterir.

In [ ]:
fig = plt.figure(figsize=(4, 3))

# Tensor'u numpy'a çevir: .cpu() → GPU'dan CPU'ya al, .detach() → gradient bağlantısını kes
codings_np = codings.cpu().detach().numpy()

# Latent space'deki tüm noktaları mavi nokta olarak çiz
plt.plot(codings_np[:, 0], codings_np[:, 1], "b.")

# Eksen etiketleri: z₁ ve z₂ latent boyutları
plt.xlabel("$z_1$", fontsize=18)
plt.ylabel("$z_2$", fontsize=18, rotation=0)
plt.grid(True)

plt.show()

# 3. Stacked Autoencoders (Derin Katmanlı Otokodlayıcılar)

**Stacked Autoencoder**, birden fazla gizli katman içeren derin bir autoencoder mimarisidir. Her katman daha soyut özellikler öğrenir.

```
Giriş(784) → 128 → 32  (Encoder — sıkıştırma)
              ↓
             32 (Latent Space / Bottleneck)
              ↓
        32 → 128 → 784 (Decoder — genişletme)
```

Aktivasyon fonksiyonu olarak **ReLU** (encoder) ve **Sigmoid** (decoder son katman) kullanıyoruz.
- **ReLU**: Ara katmanlarda negatif değerleri sıfırlar, hızlı öğrenme
- **Sigmoid**: Son decoder katmanında, piksel değerlerini [0,1] aralığına sıkıştırır (görüntü piksel aralığı)

Veri seti olarak **Fashion MNIST** kullanıyoruz: 70.000 adet 28×28 piksel giysi görüntüsü (10 sınıf).


## 3.1 Stacked Autoencoder Mimarisini Oluşturma

In [ ]:
torch.manual_seed(42)

# ── ENCODER ────────────────────────────────────────────────────────────────
stacked_encoder = nn.Sequential(
    nn.Flatten(),                           # 1×28×28 → 784 (düzleştir)
    nn.Linear(1 * 28 * 28, 128), nn.ReLU(),# 784 → 128 nöron, ReLU aktivasyon
    nn.Linear(128, 32), nn.ReLU(),          # 128 → 32 nöron (bottleneck), ReLU
)
# Encoder çıkışı: 32 boyutlu latent vector (784 boyutun 32'ye sıkıştırılması!)

# ── DECODER ────────────────────────────────────────────────────────────────
stacked_decoder = nn.Sequential(
    nn.Linear(32, 128), nn.ReLU(),          # 32 → 128 nöron, genişletme başlıyor
    nn.Linear(128, 1 * 28 * 28), nn.Sigmoid(),  # 128 → 784 nöron, Sigmoid [0,1]
    nn.Unflatten(dim=1, unflattened_size=(1, 28, 28))  # 784 → 1×28×28 şekline döndür
)
# Decoder çıkışı: 1×28×28 boyutunda görüntü tensoru

# ── TAM AUTOENCODER ────────────────────────────────────────────────────────
stacked_ae = nn.Sequential(stacked_encoder, stacked_decoder).to(device)

## 3.2 Fashion MNIST Veri Setini Yükleme

Fashion MNIST, Zalando'nun giysi görüntülerinden oluşan bir veri setidir. MNIST rakam veri setinin daha zorlu bir alternatifidir.

- **60.000** eğitim + **10.000** test görüntüsü
- Her görüntü: **28×28 piksel, gri tonlamalı**
- **10 sınıf**: T-shirt, Trouser, Pullover, Dress, Coat, Sandal, Shirt, Sneaker, Bag, Ankle boot

Veriyi **55.000 train / 5.000 validation** olarak böleceğiz.

In [ ]:
import torchvision
import torchvision.transforms.v2 as T  # Yeni stil transforms API

# Dönüşüm pipeline'ı tanımla:
# 1. ToImage(): PIL Image veya numpy array'i torch tensor'a çevir
# 2. ToDtype(float32, scale=True): uint8 [0,255] → float32 [0.0, 1.0]
toTensor = T.Compose([T.ToImage(), T.ToDtype(torch.float32, scale=True)])

# Fashion MNIST veri setini indir ve yükle
train_and_valid_data = torchvision.datasets.FashionMNIST(
    root="datasets",    # Veriyi kaydedeceğimiz klasör
    train=True,         # Eğitim seti (60.000 görüntü)
    download=True,      # İndir (eğer yoksa)
    transform=toTensor  # Yukarıdaki dönüşümü uygula
)
test_data = torchvision.datasets.FashionMNIST(
    root="datasets",
    train=False,        # Test seti (10.000 görüntü)
    download=True,
    transform=toTensor
)

# Eğitim setini rastgele böl: 55.000 train + 5.000 validation
torch.manual_seed(42)
train_data, valid_data = torch.utils.data.random_split(
    train_and_valid_data, [55_000, 5_000])

## 3.3 AutoencoderDataset Wrapper Sınıfı

Fashion MNIST normalde `(görüntü, etiket)` çiftleri döndürür. Ancak autoencoder eğitiminde etikete ihtiyacımız yok; hem giriş hem de hedef aynı görüntü olmalı. Bu nedenle özel bir `Dataset` wrapper'ı yazıyoruz.

In [ ]:
from torch.utils.data import Dataset

class AutoencoderDataset(Dataset):
    """
    Herhangi bir (X, label) dataset'ini autoencoder eğitimi için
    (X, X) formatına dönüştüren wrapper sınıfı.
    
    Autoencoder'lar gözetimsiz öğrenme yapar:
    Giriş = Hedef (modelin görüntüyü yeniden oluşturmasını istiyoruz)
    """
    def __init__(self, base_dataset):
        self.base_dataset = base_dataset

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, idx):
        x, _ = self.base_dataset[idx]  # Etiketi görmezden gel (_)
        return x, x  # Hem giriş hem hedef olarak aynı görüntüyü döndür

# DataLoader'ları oluştur
train_loader = DataLoader(AutoencoderDataset(train_data), batch_size=32,
                          shuffle=True)   # Eğitim: her epoch'ta karıştır
valid_loader = DataLoader(AutoencoderDataset(valid_data), batch_size=32)
test_loader  = DataLoader(AutoencoderDataset(test_data),  batch_size=32)

## 3.4 Modeli Eğitme

In [ ]:
# Stacked autoencoder'ı eğit
optimizer = torch.optim.NAdam(stacked_ae.parameters(), lr=0.01)
mse = nn.MSELoss()
rmse = torchmetrics.MeanSquaredError(squared=False).to(device)

# 10 epoch — Fashion MNIST büyük bir veri seti, bu yüzden daha az epoch yeterli
history = train(stacked_ae, optimizer, mse, rmse, train_loader, valid_loader,
                n_epochs=10)

## 3.5 Yeniden Oluşturma (Reconstruction) Görselleştirme

Aşağıdaki iki yardımcı fonksiyon, orijinal görüntüler ve autoencoder'ın bunları nasıl yeniden oluşturduğunu yan yana gösterir. Bu sayede modelin ne kadar iyi öğrendiğini görsel olarak değerlendiririz.

In [ ]:
def plot_image(image):
    """
    Tek bir görüntüyü gösterir.
    
    .permute(1, 2, 0): PyTorch formatı [C, H, W] → Matplotlib formatı [H, W, C]
    cmap='binary': Gri tonlamalı görüntü
    """
    plt.imshow(image.permute(1, 2, 0).cpu(), cmap="binary")
    plt.axis("off")  # Eksen numaralarını gizle


def plot_reconstructions(model, images, n_images=5):
    """
    Orijinal görüntüleri üst sırada, modelin yeniden oluşturduklarını alt sırada gösterir.
    
    Args:
        model: Autoencoder modeli
        images: Görüntü tensoru [N, C, H, W]
        n_images: Gösterilecek görüntü sayısı
    """
    images = images[:n_images]  # İlk n_images görüntüyü al
    
    with torch.no_grad():  # Gradient hesaplamayı kapat
        y_pred = model(images.to(device))
    
    # Named tuple desteği (VAE, SparseAE gibi modeller için)
    if isinstance(y_pred, tuple):
        y_pred = y_pred.output
    
    fig = plt.figure(figsize=(len(images) * 1.5, 3))
    
    for idx in range(len(images)):
        # Üst sıra: orijinal görüntüler
        plt.subplot(2, len(images), 1 + idx)
        plot_image(images[idx])
        
        # Alt sıra: yeniden oluşturulan görüntüler
        plt.subplot(2, len(images), 1 + len(images) + idx)
        plot_image(y_pred[idx])

In [ ]:
# Validation setinden görüntüleri al
X_valid = torch.stack([x for x, _ in valid_data])

# Orijinal ve yeniden oluşturulan görüntüleri yan yana göster
plot_reconstructions(stacked_ae, X_valid)

plt.show()
# Üst sıra: orijinal Fashion MNIST görüntüleri
# Alt sıra: autoencoder'ın ürettiği (yeniden oluşturulmuş) görüntüler
# Görüntüler biraz bulanık çünkü 784 boyut → 32 boyuta sıkıştırıldı

## 3.6 Anomaly Detection (Anormallik Tespiti)

Autoencoder'ların güçlü bir uygulaması **anomaly detection**'dır.

**Temel Fikir**: 
- Model yalnızca **normal** veri (Fashion MNIST) üzerinde eğitildi
- Eğitim sırasında görmediği veriler (MNIST rakamları) için **yeniden oluşturma kaybı yüksek** olur
- Bu yüksek kayıp, veri noktasının "anormal" olduğuna işaret eder

Örnek kullanım alanları:
- Ürün kalite kontrolü (hatalı parçaları tespit)
- Sahtekarlık tespiti (banka işlemlerinde)
- Ağ güvenliği (anormal trafik tespiti)


In [ ]:
# MNIST veri setini yükle (Fashion MNIST ile aynı transform)
torch.manual_seed(42)
mnist_data = torchvision.datasets.MNIST(
    root="datasets", train=True, download=True, transform=toTensor)

# Validation setiyle aynı sayıda MNIST görüntüsü al
mnist_images = torch.stack([mnist_data[i][0] for i in range(X_valid.size(0))])

# Stacked autoencoder'ın MNIST görüntülerini nasıl "yeniden oluşturduğuna" bak
# Beklenti: Fashion MNIST'e göre eğitildiğinden MNIST görüntülerini kötü yeniden oluşturmalı
plot_reconstructions(stacked_ae, images=mnist_images)

plt.show()

In [ ]:
# MNIST görüntüleri için toplam reconstruction loss hesapla
images = mnist_images.to(device)
with torch.no_grad():
    y_pred = stacked_ae(images)
    # Tüm görüntüler için ortalama MSE
    recon_loss = torch.nn.functional.mse_loss(y_pred, images)

recon_loss  # MNIST için reconstruction loss yüksek olmalı!

In [ ]:
import torch.nn.functional as F

def compute_reconstruction_losses(X, device):
    """
    Her görüntü için ayrı ayrı reconstruction loss hesaplar.
    
    Returns:
        Her görüntünün MSE kaybını içeren 1D tensor
    """
    X = X.to(device)
    with torch.no_grad():
        y_pred = stacked_ae(X)
        # reduction='none': Her örnek için ayrı kayıp (ortalama almadan)
        # .view(X.size(0), -1): [N, 1, 28, 28] → [N, 784]
        # .mean(dim=1): Her görüntünün 784 pikseli için ortalama al → [N]
        return F.mse_loss(y_pred, X, reduction="none").view(X.size(0), -1).mean(dim=1).cpu()

# Her iki veri seti için kayıp dağılımını hesapla
recon_losses_mnist        = compute_reconstruction_losses(mnist_images, device)
recon_losses_fashion_mnist = compute_reconstruction_losses(X_valid, device)

# Histogram olarak karşılaştır
plt.hist(recon_losses_mnist, bins=85, alpha=0.8, label="MNIST images")
plt.hist(recon_losses_fashion_mnist, bins=85, alpha=0.8, label="Fashion MNIST images")
plt.xlabel("Reconstruction loss (MSE)")
plt.ylabel("Count", rotation=0)
plt.legend()
plt.grid()

plt.show()
# Beklenti: MNIST görüntüleri için kayıp dağılımı sağa kayık (daha yüksek kayıp)
# Fashion MNIST görüntüleri için kayıp dağılımı sola kayık (daha düşük kayıp)

## 3.7 Fashion MNIST Veri Setini t-SNE ile Görselleştirme

**t-SNE** (t-Distributed Stochastic Neighbor Embedding), yüksek boyutlu veriyi 2D/3D'ye indirgeyen güçlü bir görselleştirme tekniğidir. Benzer örnekleri birbirine yakın, farklı örnekleri uzak yerleştirir.

Burada encoder'ın ürettiği **32 boyutlu** coding'leri t-SNE ile 2D'ye indirgiyor ve sınıflara göre renklendirerek görüntülüyoruz.

In [ ]:
from sklearn.manifold import TSNE

# Validation setindeki tüm görüntüleri encoder'dan geçirip 32 boyutlu coding'lere çevir
with torch.no_grad():
    X_valid_compressed = stacked_encoder(X_valid.to(device))
# X_valid_compressed.shape: [5000, 32]

# t-SNE ile 32 boyuttan 2 boyuta indir
# init='pca': PCA ile başlat (daha iyi ve tekrarlanabilir sonuçlar)
# learning_rate='auto': Otomatik learning rate seçimi
tsne = TSNE(init="pca", learning_rate="auto", random_state=42)
X_valid_2D = tsne.fit_transform(X_valid_compressed.cpu())
# X_valid_2D.shape: [5000, 2]

In [ ]:
# Validation setinin etiketlerini al
y_valid = torch.tensor([y for _, y in valid_data])

# 2D scatter plot: her nokta bir görüntü, rengi sınıf etiketi
plt.scatter(X_valid_2D[:, 0], X_valid_2D[:, 1], c=y_valid, s=10, cmap="tab10")
plt.show()
# 10 farklı renk = 10 farklı giysi sınıfı
# Birbirine yakın aynı renkli noktalar: benzer görüntüler latent space'de birbirine yakın!

In [ ]:
# Kitap için güzelleştirilmiş versiyon: noktaların üzerine gerçek görüntüler yerleştir
import matplotlib as mpl

plt.figure(figsize=(10, 8))
cmap = plt.cm.tab10

# Koordinatları [0,1] aralığına normalize et
Z = X_valid_2D
Z = (Z - Z.min()) / (Z.max() - Z.min())

plt.scatter(Z[:, 0], Z[:, 1], c=y_valid, s=10, cmap=cmap)

# Birbirine çok yakın görüntüleri atlayarak her görüntünün koordinatına
# küçük bir thumbnail (küçük resim) yerleştir
image_positions = np.array([[1., 1.]])
for index, position in enumerate(Z):
    # Bu konumdan en yakın görüntüye olan mesafeyi hesapla
    dist = ((position - image_positions) ** 2).sum(axis=1)
    if dist.min() > 0.02:  # Yeterince uzaksa görüntüyü göster
        image_positions = np.r_[image_positions, [position]]
        imagebox = mpl.offsetbox.AnnotationBbox(
            mpl.offsetbox.OffsetImage(X_valid[index].squeeze(dim=0), cmap="binary"),
            position,
            bboxprops={"edgecolor": cmap(y_valid[index]), "lw": 2})
        plt.gca().add_artist(imagebox)

plt.axis("off")
plt.show()

## 3.8 Tied Weights Autoencoder

**Tied Weights** (Bağlı Ağırlıklar): Decoder ağırlıklarının encoder ağırlıklarının **transpozu** olarak kullanıldığı bir tekniktir.

**Neden kullanılır?**
- **Parametre sayısını azaltır**: Her iki katman için ayrı ağırlık yerine paylaşılan ağırlıklar
- **Overfitting'i azaltır**: Daha az parametre = daha iyi genelleme
- **Simetri kısıtı**: Encoder ve decoder birbirinin tersi olmak zorunda kalır

**Uygulama**:
- `enc1.weight` (784→128) → decoder'da `enc1.weight.T` (128→784) kullanılır
- Bias'lar paylaşılmaz (ayrı `dec1_bias`, `dec2_bias` parametreleri)


In [ ]:
import torch.nn.functional as F

class TiedAutoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        # Encoder katmanları: ağırlıklar burada tanımlanır
        self.enc1 = nn.Linear(1 * 28 * 28, 128)  # 784 → 128
        self.enc2 = nn.Linear(128, 32)            # 128 → 32

        # Decoder için sadece bias parametreleri gerekiyor
        # Çünkü ağırlıkları encoder'dan alacağız (transpose)
        self.dec1_bias = nn.Parameter(torch.zeros(128))        # 32 → 128 için bias
        self.dec2_bias = nn.Parameter(torch.zeros(1 * 28 * 28))  # 128 → 784 için bias

    def encode(self, X):
        """3D tensor'u önce düzleştirir, sonra 2 linear katmandan geçirir."""
        Z = X.view(-1, 1 * 28 * 28)  # [B, 1, 28, 28] → [B, 784] (flatten)
        Z = F.relu(self.enc1(Z))      # [B, 784] → [B, 128], ReLU
        return F.relu(self.enc2(Z))   # [B, 128] → [B, 32], ReLU

    def decode(self, X):
        """
        Encoder ağırlıklarının TRANSPOSE'unu kullanarak decoder işlemi yapar.
        F.linear(input, weight, bias) = input @ weight.T + bias
        enc2.weight: [32, 128] → enc2.weight.t(): [128, 32] — 32→128 geçiş
        enc1.weight: [128, 784] → enc1.weight.t(): [784, 128] — 128→784 geçiş
        """
        Z = F.relu(F.linear(X, self.enc2.weight.t(), self.dec1_bias))    # 32 → 128
        Z = F.sigmoid(F.linear(Z, self.enc1.weight.t(), self.dec2_bias)) # 128 → 784
        return Z.view(-1, 1, 28, 28)  # [B, 784] → [B, 1, 28, 28]

    def forward(self, X):
        return self.decode(self.encode(X))

In [ ]:
# Tied autoencoder'ı eğit
tied_ae = TiedAutoencoder().to(device)
optimizer = torch.optim.NAdam(tied_ae.parameters(), lr=0.01)
history = train(tied_ae, optimizer, mse, rmse, train_loader, valid_loader, n_epochs=10)

In [ ]:
# Tied autoencoder rekonstrüksiyonlarını göster
plot_reconstructions(tied_ae, X_valid)
plt.show()

## 3.9 Convolutional Autoencoder (Evrişimli Otokodlayıcı)

**Convolutional Autoencoder**, görüntü verisi için özellikle uygun olan bir mimaridir. Tam bağlı katmanlar yerine **Conv2d** (encoder) ve **ConvTranspose2d** (decoder) katmanları kullanır.

**Avantajları**:
- Uzamsal (spatial) ilişkileri daha iyi yakalar
- Parametre paylaşımı sayesinde daha verimli
- Translasyon değişmezliği (translation invariance)

**Encoder**: Conv → MaxPool → Conv → MaxPool → Conv → MaxPool → AdaptiveAvgPool → Flatten
**Decoder**: Linear → Unflatten → ConvTranspose → ConvTranspose → ConvTranspose

`ConvTranspose2d` (Deconvolution / Upsampling): Boyutu büyüten convolution işlemi.


In [ ]:
torch.manual_seed(42)

# ── ENCODER ────────────────────────────────────────────────────────────────
conv_encoder = nn.Sequential(
    # 1. Conv blok: 1 kanal → 16 özellik haritası, aynı boyut korunur
    nn.Conv2d(1, 16, kernel_size=3, padding="same"), nn.ReLU(),
    # MaxPool: 28×28 → 14×14 (boyutu yarıya indir)
    nn.MaxPool2d(kernel_size=2),           # çıkış: 16 × 14 × 14
    
    # 2. Conv blok: 16 → 32 özellik haritası
    nn.Conv2d(16, 32, kernel_size=3, padding="same"), nn.ReLU(),
    # MaxPool: 14×14 → 7×7
    nn.MaxPool2d(kernel_size=2),           # çıkış: 32 × 7 × 7
    
    # 3. Conv blok: 32 → 64 özellik haritası
    nn.Conv2d(32, 64, kernel_size=3, padding="same"), nn.ReLU(),
    # MaxPool: 7×7 → 3×3
    nn.MaxPool2d(kernel_size=2),           # çıkış: 64 × 3 × 3
    
    # 4. Conv blok: 64 → 32 özellik haritası (sıkıştır)
    nn.Conv2d(64, 32, kernel_size=3, padding="same"), nn.ReLU(),
    # AdaptiveAvgPool: herhangi bir boyutu → 1×1 (global average pooling)
    nn.AdaptiveAvgPool2d((1, 1)),          # çıkış: 32 × 1 × 1
    nn.Flatten()                           # çıkış: 32 boyutlu vektör
)

# ── DECODER ────────────────────────────────────────────────────────────────
conv_decoder = nn.Sequential(
    # 32 → 16×3×3 = 144 nöron
    nn.Linear(32, 16 * 3 * 3),
    # 1D → 3D: [B, 144] → [B, 16, 3, 3]
    nn.Unflatten(dim=1, unflattened_size=(16, 3, 3)),
    
    # ConvTranspose: 3×3 → 7×7 (stride=2 ile boyutu büyüt)
    nn.ConvTranspose2d(16, 32, kernel_size=3, stride=2), nn.ReLU(),
    
    # ConvTranspose: 7×7 → 14×14
    nn.ConvTranspose2d(32, 16, kernel_size=3, stride=2, padding=1,
                       output_padding=1), nn.ReLU(),
    
    # ConvTranspose: 14×14 → 28×28, Sigmoid ile [0,1] aralığına normalize
    nn.ConvTranspose2d(16, 1, kernel_size=3, stride=2, padding=1,
                       output_padding=1), nn.Sigmoid()
)

conv_ae = nn.Sequential(conv_encoder, conv_decoder).to(device)

In [ ]:
# Convolutional autoencoder'ı eğit
optimizer = torch.optim.NAdam(conv_ae.parameters(), lr=0.005)
history = train(conv_ae, optimizer, mse, rmse, train_loader, valid_loader,
                n_epochs=10)

In [ ]:
# Rekonstrüksiyonları göster
plot_reconstructions(conv_ae, X_valid)
plt.show()

# 4. Recurrent Autoencoder (Tekrarlayan Sinir Ağlı Otokodlayıcı) — Ekstra Materyal

**Recurrent Autoencoder**, sıralı (sequential) veri için tasarlanmış bir autoencoder mimarisidir. Görüntüleri her satırı 28 boyutlu bir vektör olan **28 adımlık bir sekans** olarak işler.

**Mimari**:
- **Encoder LSTM**: 28×28 görüntüyü sekans olarak işler, son hidden state'i coding'e project eder
- **Decoder LSTM**: Coding'i 28 kez tekrarlayarak sekans üretir

**Ne zaman kullanılır?**
- Zaman serisi verisi
- Doğal dil işleme
- Sinyal işleme


In [ ]:
class RecurrentAutoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        
        # ── ENCODER ──────────────────────────────────────────────────────────
        # 2 katmanlı LSTM: her adımda 28 özellik giriyor, 128 hidden unit çıkıyor
        # batch_first=True: [batch, sequence, features] formatı
        self.encoder_lstm = nn.LSTM(input_size=28, hidden_size=128,
                                    num_layers=2, batch_first=True)
        # LSTM'nin 128 boyutlu son hidden state'ini 32 boyutlu latent vector'a sıkıştır
        self.encoder_proj = nn.Linear(128, 32)

        # ── DECODER ──────────────────────────────────────────────────────────
        # LSTM: 32 boyutlu coding'i 28 adım boyunca işler
        self.decoder_lstm = nn.LSTM(input_size=32, hidden_size=128,
                                    batch_first=True)
        # Her adımın 128 boyutlu çıktısını 28 boyuta project et (bir görüntü satırı)
        self.decoder_proj = nn.Linear(128, 28)

    def encode(self, X):
        """
        X shape: [B, 1, 28, 28] (batch, kanal, yükseklik, genişlik)
        """
        Z = X.squeeze(dim=1)         # [B, 1, 28, 28] → [B, 28, 28] (kanal boyutunu kaldır)
        _, (h_n, _) = self.encoder_lstm(Z)  # h_n shape: [n_layers=2, B, 128]
        Z = h_n[-1]                  # Son LSTM katmanının hidden state: [B, 128]
        return self.encoder_proj(Z)  # [B, 128] → [B, 32] (latent coding)

    def decode(self, X):
        """
        X shape: [B, 32] (latent coding)
        """
        # Coding'i 28 kez tekrarla (görüntünün 28 satırı için)
        Z = X.unsqueeze(dim=1).repeat(1, 28, 1)  # [B, 1, 32] → [B, 28, 32]
        Z, _ = self.decoder_lstm(Z)              # [B, 28, 128]
        # Her adımın çıktısını 28 boyuta project et ve Sigmoid uygula
        return F.sigmoid(self.decoder_proj(Z).unsqueeze(dim=1))  # [B, 1, 28, 28]

    def forward(self, X):
        return self.decode(self.encode(X))

torch.manual_seed(42)
recurrent_ae = RecurrentAutoencoder().to(device)

In [ ]:
# Recurrent autoencoder'ı eğit (daha küçük lr, LSTM eğitimi hassas)
optimizer = torch.optim.NAdam(recurrent_ae.parameters(), lr=1e-3)
history = train(recurrent_ae, optimizer, mse, rmse, train_loader, valid_loader,
                n_epochs=10)

In [ ]:
# Rekonstrüksiyonları göster
plot_reconstructions(recurrent_ae, X_valid)
plt.show()

# 5. Denoising Autoencoders (Gürültü Giderici Otokodlayıcılar)

**Denoising Autoencoder**: Giriş görüntüsüne kasıtlı olarak gürültü ekleyip, modelin gürültüsüz orijinali yeniden oluşturmasını öğrettiğimiz bir autoencoder türü.

**Eğitim Süreci**:
```
Orijinal görüntü → [Gürültü Ekle] → Bozulmuş görüntü → [Encoder] → Coding → [Decoder] → Orijinal görüntü
```

**Avantajları**:
- Daha güçlü ve genellenebilir özellikler öğrenir
- Overfitting'e karşı daha dirençli
- Gerçek dünya uygulamalarında gürültü giderme (image denoising)

**İki Yöntem**:
1. **Dropout**: Rastgele nöronları sıfırlama
2. **Gaussian Noise**: Sürekli değerli Gaussian gürültüsü ekleme


## 5.1 Dropout ile Denoising Autoencoder

`nn.Dropout(0.5)` encoder'ın başına eklenerek giriş özelliklerinin %50'si sıfırlanır. Model, eksik bilgiyle görüntüyü tamamlamayı öğrenir.

In [ ]:
torch.manual_seed(42)

# ── ENCODER (Dropout ile) ─────────────────────────────────────────────────
dropout_encoder = nn.Sequential(
    nn.Flatten(),
    # Giriş piksellerinin %50'sini rastgele sıfırla (sadece eğitimde)
    # Bu sayede model eksik bilgiyle görüntüyü tamamlamayı öğrenir
    nn.Dropout(0.5),
    nn.Linear(1 * 28 * 28, 128), nn.ReLU(),
    nn.Linear(128, 128), nn.ReLU(),
)

# ── DECODER ──────────────────────────────────────────────────────────────
dropout_decoder = nn.Sequential(
    nn.Linear(128, 128), nn.ReLU(),
    nn.Linear(128, 1 * 28 * 28), nn.Sigmoid(),
    nn.Unflatten(dim=1, unflattened_size=(1, 28, 28))
)

dropout_ae = nn.Sequential(dropout_encoder, dropout_decoder).to(device)

In [ ]:
# Dropout autoencoder'ı eğit
optimizer = torch.optim.NAdam(dropout_ae.parameters(), lr=0.01)
history = train(dropout_ae, optimizer, mse, rmse, train_loader, valid_loader,
                n_epochs=10)

In [ ]:
# Dropout ile bozulmuş görüntüleri yeniden oluştur
# Not: dropout(X_valid) → görüntülerin %50 pikseli sıfırlanmış
torch.manual_seed(42)
dropout = nn.Dropout(0.5)
plot_reconstructions(dropout_ae, dropout(X_valid))

plt.show()
# Üst sıra: bozulmuş (piksellerin %50'si sıfır) görüntüler
# Alt sıra: model bu bozulmuş görüntülerden orijinali kurtarıyor!

## 5.2 Gaussian Noise ile Denoising Autoencoder

Dropout yerine her piksele küçük Gaussian gürültü ekliyoruz. Daha gerçekçi bir gürültü modeli.

In [ ]:
class GaussianNoise(nn.Module):
    """
    Eğitim sırasında giriş veriye Gaussian (normal dağılımlı) gürültü ekleyen katman.
    Değerlendirme sırasında (eval mode) hiçbir gürültü eklenmez.
    
    Args:
        std: Gaussian gürültünün standart sapması
    """
    def __init__(self, std):
        super().__init__()
        self.std = std

    def forward(self, X):
        if self.training:  # Sadece eğitim modunda gürültü ekle
            # X ile aynı şekilde Gaussian gürültü üret ve ekle
            noise = torch.randn_like(X) * self.std
            return X + noise
        return X  # Değerlendirme modunda gürültüsüz döndür

In [ ]:
torch.manual_seed(42)

# ── ENCODER (Gaussian Noise ile) ─────────────────────────────────────────
noise_encoder = nn.Sequential(
    nn.Flatten(),
    # std=0.5: Görüntü piksel değerlerine [0,1] aralığında standart sapması 0.5
    # olan Gaussian gürültü ekle (eğitim modunda)
    GaussianNoise(0.5),
    nn.Linear(1 * 28 * 28, 128), nn.ReLU(),
    nn.Linear(128, 128), nn.ReLU(),
)

# ── DECODER ──────────────────────────────────────────────────────────────
noise_decoder = nn.Sequential(
    nn.Linear(128, 128), nn.ReLU(),
    nn.Linear(128, 1 * 28 * 28), nn.Sigmoid(),
    nn.Unflatten(dim=1, unflattened_size=(1, 28, 28))
)

noise_ae = nn.Sequential(noise_encoder, noise_decoder).to(device)

In [ ]:
# Gaussian noise autoencoder'ı eğit
optimizer = torch.optim.NAdam(noise_ae.parameters(), lr=0.01)
history = train(noise_ae, optimizer, mse, rmse, train_loader, valid_loader,
                n_epochs=10)

In [ ]:
# Gaussian gürültülü görüntüleri yeniden oluştur
torch.manual_seed(42)
noise = GaussianNoise(0.5)
# noise(X_valid): eval modda olduğumuzdan GaussianNoise sınıfı gürültü eklemez!
# Bu yüzden doğrudan gürültü ekleyerek test ediyoruz:
noise.training = True  # Manuel olarak training moduna al
plot_reconstructions(noise_ae, noise(X_valid))
plt.show()

# 6. Sparse Autoencoder (Seyrek Otokodlayıcı)

**Sparse Autoencoder**: Coding katmanındaki aktivasyonların büyük çoğunluğunun sıfır (veya sıfıra yakın) olmasını zorlamak için regularizasyon kullanan bir autoencoder.

**Seyreklik Neden İstenir?**
- Her girdi için yalnızca birkaç nöron aktive olursa, her nöron belirli bir özelliği temsil etmeye zorlanır
- Bu, **yorumlanabilir** (interpretable) temsillerle sonuçlanabilir
- Her nöron farklı bir görsel özelliğe (kenar, doku, renk) karşılık gelebilir

**KL Divergence Regularizasyonu**:
İstenen seyreklik `p` (örn. %10) ile gerçek ortalama aktivasyon `q` arasındaki KL divergence'ı minimize ederiz:

```
KL(p || q) = p * log(p/q) + (1-p) * log((1-p)/(1-q))
```

`q` tam olarak `p`'ye eşit olduğunda KL=0 (hiç penalty yok).


In [ ]:
from collections import namedtuple

# Named tuple: model hem çıktıyı hem de coding'i döndürecek
# Böylece loss fonksiyonunda coding'e erişebiliriz
AEOutput = namedtuple("AEOutput", ["output", "codings"])

class SparseAutoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        
        # ── ENCODER ────────────────────────────────────────────────────────
        self.encoder = nn.Sequential(
            nn.Flatten(),
            nn.Linear(1 * 28 * 28, 128), nn.ReLU(),
            # Coding: 256 nöron (çok fazla!) ama Sigmoid ile [0,1] aralığına çek
            # KL regularization bu nöronların çoğunu 0'a yakın tutmaya zorlayacak
            nn.Linear(128, 256), nn.Sigmoid()
        )
        
        # ── DECODER ────────────────────────────────────────────────────────
        self.decoder = nn.Sequential(
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, 1 * 28 * 28), nn.Sigmoid(),
            nn.Unflatten(dim=1, unflattened_size=(1, 28, 28))
        )

    def forward(self, X):
        codings = self.encoder(X)   # [B, 256] — seyrek aktivasyonlar
        output  = self.decoder(codings)
        return AEOutput(output, codings)  # Named tuple olarak döndür

In [ ]:
def mse_plus_sparsity_loss(y_pred, y_target, target_sparsity=0.1,
                           kl_weight=1e-3, eps=1e-8):
    """
    MSE + KL Divergence seyreklik regularizasyonu içeren kayıp fonksiyonu.
    
    Args:
        y_pred: AEOutput named tuple (output, codings)
        y_target: Hedef görüntüler
        target_sparsity: İstenen seyreklik oranı (p), varsayılan %10
        kl_weight: KL divergence'ın toplam kayıptaki ağırlığı
        eps: Sayısal kararlılık için küçük sayı (log(0) hatasını önler)
    
    Returns:
        MSE reconstruction loss + KL divergence penalty
    """
    # p: İstenen seyreklik (tensor olarak cihaza taşı)
    p = torch.tensor(target_sparsity, device=y_pred.codings.device)
    
    # q: Her coding nöronunun batch üzerindeki ortalama aktivasyonu
    # .mean(dim=0): Her nöron için batch ortalaması → [256] boyutlu vektör
    # torch.clamp: 0 ve 1'e çok yakın değerleri log hesabı için güvenli aralığa çek
    q = torch.clamp(y_pred.codings.mean(dim=0), eps, 1 - eps)
    
    # KL Divergence: p*log(p/q) + (1-p)*log((1-p)/(1-q))
    # q = p olduğunda KL = 0 (mükemmel seyreklik, ceza yok)
    # q >> p veya q << p olduğunda KL büyük (ceza var)
    kl_div = p * torch.log(p / q) + (1 - p) * torch.log((1 - p) / (1 - q))
    
    return mse(y_pred.output, y_target) + kl_weight * kl_div.sum()

In [ ]:
# KL Divergence vs MAE vs MSE karşılaştırma grafiği
plt.figure(figsize=(6, 3.5))
p = 0.1        # Hedef seyreklik = %10
q = np.linspace(0.001, 0.999, 500)  # Gerçek seyreklik (0'dan 1'e)

# KL Divergence: p sabit, q değişiyor
kl_div = p * np.log(p / q) + (1 - p) * np.log((1 - p) / (1 - q))
mse_   = (p - q) ** 2   # MSE: ikinci dereceden ceza
mae    = np.abs(p - q)  # MAE: birinci dereceden ceza

# p = 0.1 konumuna dikey çizgi (hedef seyreklik)
plt.plot([p, p], [0, 0.3], "k:")
plt.text(0.05, 0.32, "Target\nsparsity $p$", fontsize=14)

plt.plot(q, kl_div, "b-",  label="KL divergence")
plt.plot(q, mae,    "g--", label=r"MAE ($\ell_1$)")
plt.plot(q, mse_,   "r--", linewidth=1, label=r"MSE ($\ell_2$)")

plt.legend(loc="upper left", fontsize=14)
plt.xlabel("Actual sparsity $q$")
plt.ylabel("Cost", rotation=0)
plt.axis([0, 1, 0, 0.95])
plt.grid(True)
# KL Divergence: q=p'de minimum (0), q=p'den uzaklaştıkça hızla artar
# Bu asimetrik ceza yapısı, KL'yi seyreklik için özellikle uygun kılar

In [ ]:
# Sparse autoencoder'ı eğit
torch.manual_seed(42)
sparse_ae = SparseAutoencoder().to(device)
optimizer = torch.optim.NAdam(sparse_ae.parameters(), lr=0.002)
history = train(sparse_ae, optimizer, mse_plus_sparsity_loss, rmse,
                train_loader, valid_loader, n_epochs=10)

In [ ]:
# Sparse autoencoder rekonstrüksiyonlarını göster
plot_reconstructions(sparse_ae, X_valid)
plt.show()

In [ ]:
# Coding aktivasyon dağılımını incele
with torch.no_grad():
    y_pred = sparse_ae(X_valid.to(device))

# Tüm coding değerlerini düzleştir
encs = y_pred.codings.flatten().detach().cpu()

# Log ölçekli histogram: seyreklik varsa 0'a yakın değerler çok fazla olmalı
plt.hist(encs, log=True)
plt.xlabel("Activation")
plt.ylabel("Count")
plt.show()
# Beklenti: Değerlerin büyük çoğunluğu 0'a yakın (seyrek!), 
# çok az sayıda nöron yüksek aktivasyon gösteriyor

In [ ]:
# Gerçek ortalama aktivasyonu hesapla
# Hedef: ~0.10 (target_sparsity=0.1)
y_pred.codings.mean()

In [ ]:
def plot_multiple_images(images, n_cols=None):
    """
    Birden fazla görüntüyü ızgara şeklinde gösterir.
    
    Args:
        images: Görüntü tensoru listesi veya [N, C, H, W] tensoru
        n_cols: Sütun sayısı (varsayılan: tüm görüntüler tek satırda)
    """
    n_cols = n_cols or len(images)
    n_rows = (len(images) - 1) // n_cols + 1
    plt.figure(figsize=(n_cols, n_rows))
    for index, image in enumerate(images):
        plt.subplot(n_rows, n_cols, index + 1)
        plot_image(image)

# Belirli bir coding boyutunun (dim=6) yüksek aktivasyona sahip görüntüleri göster
# Bu, nöron 6'nın "neyi" öğrendiğini anlamaya yarar
dim = 6
codings = y_pred.codings[:, dim].cpu()
threshold = np.percentile(codings, 90)   # Üst %10'luk eşik
selected_images = X_valid[codings > threshold]  # Eşiği aşan görüntüler
plot_multiple_images(selected_images[:35], 7)
plt.show()
# Bu görüntüler hepsi benzer bir özelliği paylaşıyor olmalı (örn. aynı sınıf veya doku)

# 7. Variational Autoencoder (VAE — Değişimsel Otokodlayıcı)

**VAE**, standart autoencoder'dan farklı olarak **latent space'i sürekli ve düzenli** bir olasılık dağılımı olarak modeller. Bu sayede latent space'den yeni noktalar örnekleyerek **tamamen yeni görüntüler üretebiliriz**.

## Temel Fark: Standart AE vs VAE

| | Standart AE | VAE |
|---|---|---|
| Coding | Tek nokta (z) | Dağılım (μ, σ²) |
| Latent Space | Düzensiz, delikli | Sürekli, Gaussian-benzeri |
| Üretim | Zor (interpolation) | Kolay (örnekleme) |

## Matematiksel Çerçeve

Encoder, her giriş için iki vektör üretir:
- **μ (mean)**: Latent dağılımın ortası
- **log(σ²) (logvar)**: Latent dağılımın genişliği (log varyans)

Sonra **reparameterization trick** ile örnekleme yapılır:
```
z = μ + ε * σ    (ε ~ N(0, 1))
```

Bu trick, gradient'in örnekleme işleminden geçmesine izin verir.

## VAE Loss Fonksiyonu
```
Loss = Reconstruction Loss + KL Divergence
     = MSE(x, x̂) + KL(q(z|x) || p(z))
```

**KL Divergence**:
```
KL = -0.5 * Σ(1 + log(σ²) - μ² - σ²)
```

Bu terim, latent dağılımı standart normal dağılıma N(0,I) yaklaştırır.


In [ ]:
VAEOutput = namedtuple("VAEOutput",
                       ["output", "codings_mean", "codings_logvar"])

class VAE(nn.Module):
    def __init__(self, codings_dim=32):
        super(VAE, self).__init__()
        self.codings_dim = codings_dim
        
        # ── ENCODER ────────────────────────────────────────────────────────
        # Son katman 2*codings_dim çıktı üretiyor:
        # İlk yarı → mean (μ), İkinci yarı → log variance (log σ²)
        self.encoder = nn.Sequential(
            nn.Flatten(),
            nn.Linear(1 * 28 * 28, 128), nn.ReLU(),
            nn.Linear(128, 2 * codings_dim)  # [B, 64]: hem mean hem logvar
        )
        
        # ── DECODER ────────────────────────────────────────────────────────
        self.decoder = nn.Sequential(
            nn.Linear(codings_dim, 128), nn.ReLU(),
            nn.Linear(128, 1 * 28 * 28), nn.Sigmoid(),
            nn.Unflatten(dim=1, unflattened_size=(1, 28, 28))
        )

    def encode(self, X):
        """
        Girişi encode eder ve (mean, logvar) çiftini döndürür.
        .chunk(2, dim=-1): Son boyutu 2 eşit parçaya böl
        """
        return self.encoder(X).chunk(2, dim=-1)  # → (mean[B,32], logvar[B,32])

    def sample_codings(self, codings_mean, codings_logvar):
        """
        Reparameterization Trick:
        z = μ + ε * σ  (ε ~ N(0,1))
        
        Neden log(σ²) kullanıyoruz?
        - Varyans her zaman pozitif olmalı; log kullanmak bunu otomatik sağlar
        - exp(0.5 * logvar) = sqrt(exp(logvar)) = σ
        """
        codings_std = torch.exp(0.5 * codings_logvar)  # σ = exp(logvar/2)
        noise = torch.randn_like(codings_std)           # ε ~ N(0,1)
        return codings_mean + noise * codings_std       # z = μ + ε*σ

    def decode(self, Z):
        return self.decoder(Z)

    def forward(self, X):
        codings_mean, codings_logvar = self.encode(X)
        codings = self.sample_codings(codings_mean, codings_logvar)  # Stochastic sampling
        output = self.decode(codings)
        return VAEOutput(output, codings_mean, codings_logvar)

In [ ]:
def vae_loss(y_pred, y_target, kl_weight=1.0):
    """
    VAE kayıp fonksiyonu: Reconstruction Loss + KL Divergence
    
    KL Divergence için kapalı form çözümü (q = N(μ,σ²), p = N(0,I) varsayımıyla):
    KL(q||p) = -0.5 * Σ(1 + log(σ²) - μ² - σ²)
    
    Bunu minimize etmek:
    - μ'yu 0'a yaklaştırır (mean)  
    - σ²'yi 1'e yaklaştırır (varyans)
    → Latent space standart normal dağılıma yaklaşır
    
    Args:
        y_pred: VAEOutput(output, codings_mean, codings_logvar)
        y_target: Hedef görüntüler
        kl_weight: KL divergence ağırlığı (β-VAE için β parametresi)
    """
    output, mean, logvar = y_pred
    
    # KL divergence: her batch örneği için toplam KL
    # dim=-1: coding boyutu üzerinde topla → [B] boyutlu vektör
    kl_div = -0.5 * torch.sum(1 + logvar - logvar.exp() - mean.square(), dim=-1)
    
    # MSE reconstruction loss + normalize edilmiş KL divergence
    # /784: Piksel başına normalize (reconstruction loss ile aynı ölçeğe getir)
    return F.mse_loss(output, y_target) + kl_weight * kl_div.mean() / 784

In [ ]:
# VAE'yi eğit
torch.manual_seed(42)
vae = VAE().to(device)
optimizer = torch.optim.NAdam(vae.parameters(), lr=1e-3)
history = train(vae, optimizer, vae_loss, rmse, train_loader, valid_loader,
                n_epochs=20)

In [ ]:
# VAE rekonstrüksiyonlarını göster
plot_reconstructions(vae, X_valid)
plt.show()

## 7.1 Yeni Fashion Görüntüleri Üretme

VAE'nin gücü: latent space'den **rastgele nokta örnekleyerek** daha önce hiç görmediğimiz yeni görüntüler üretebiliriz. Bu standart bir autoencoder ile mümkün değildir.

In [ ]:
torch.manual_seed(42)

# Değerlendirme moduna geç (dropout, batch norm vb. devre dışı)
vae.eval()

# Latent space'den 21 rastgele nokta örnekle: z ~ N(0, I)
# torch.randn: standart normal dağılımdan örnekleme
codings = torch.randn(3 * 7, vae.codings_dim, device=device)  # [21, 32]

with torch.no_grad():
    # Sadece decoder'ı çalıştır: latent kodları → görüntüler
    images = vae.decode(codings)

In [ ]:
# Üretilen 21 görüntüyü 3 satır × 7 sütun ızgara şeklinde göster
plot_multiple_images(images, 7)
plt.show()
# Bu görüntüler tamamen hayali! VAE bunları hiç görmemişti.

## 7.2 Semantik Interpolasyon (Semantic Interpolation)

VAE'nin düzenli latent space'i sayesinde iki görüntü arasında **yumuşak geçiş** (interpolasyon) yapabiliriz. Bir görüntüden diğerine geçerken ara görüntüler anlamlı görünür.

In [ ]:
# Latent coding'lerin boyutunu kontrol et
codings.shape  # [21, 32]

In [ ]:
torch.manual_seed(111)

# İki başlangıç noktası örnekle (başlangıç ve bitiş görüntüleri)
codings = torch.randn(2, vae.codings_dim)  # 2 adet 32 boyutlu nokta

n_images = 7  # Aralarında kaç ara görüntü üretelim

# Lineer interpolasyon ağırlıkları: 0'dan 1'e 7 eşit adım
# weights[0]=0 → tamamen ilk görüntü
# weights[6]=1 → tamamen ikinci görüntü
weights = torch.linspace(0, 1, n_images).view(n_images, 1)

# torch.lerp: Doğrusal enterpolasyon → start + weight * (end - start)
# Her ağırlık için ara nokta hesapla
codings = torch.lerp(codings[0], codings[1], weights)  # [7, 32]

with torch.no_grad():
    images = vae.decode(codings.to(device))

In [ ]:
# 7 ara görüntüyü göster: soldan sağa yumuşak geçiş olmalı
plot_multiple_images(images)
plt.show()

# 8. Discrete VAE (Kesikli Değişimsel Otokodlayıcı)

**Discrete VAE**, latent space'in **sürekli (continuous)** değil **kesikli (discrete)** olduğu bir VAE türüdür. Latent değişkenler tamsayı indeksler olarak temsil edilir.

## Motivasyon
- Bazı kavramlar doğal olarak kesiklidir (kategoriler, semboller)
- VQ-VAE ve DALL-E gibi modern modellerin temeli
- Sürekli değil, kategorik latent değişkenler

## Gumbel-Softmax Trick

Kesikli örnekleme geri yayılım (backpropagation) ile uyumlu değildir. **Gumbel-Softmax** bunu çözer:

1. Logitlerden Gumbel gürültüsü örnekle
2. Temperature τ ile softmax uygula
3. **Hard=True**: İleri geçişte one-hot, geri geçişte yumuşak (straight-through estimator)

**Temperature (τ) Annealing**: Eğitim başında τ=1 (yumuşak), sonunda τ→0 (sert/kesikli)


In [ ]:
def gumbel_softmax(logits, tau=1, hard=False, dim=-1):
    """
    Gumbel-Softmax reparameterization trick.
    
    MPS (Apple Silicon) cihazlarda PyTorch'un F.gumbel_softmax'ında bir bug var.
    Bu fonksiyon MPS için düzeltilmiş versiyonu, diğer cihazlar için orijinali kullanır.
    
    Args:
        logits: Ham sınıf logitleri [B, coding_length, n_codes]
        tau: Temperature parametresi. Düşük τ → daha keskin (sert) dağılım
        hard: True → one-hot forward, soft backward (straight-through estimator)
        dim: Softmax'ın uygulanacağı boyut
    """
    if device != "mps":
        # MPS dışı cihazlarda PyTorch'un kendi implementasyonunu kullan
        return F.gumbel_softmax(logits, tau, hard, dim)
    
    # MPS için elle Gumbel-Softmax implementasyonu:
    # Gumbel(0,1) dağılımından örnekle: -log(-log(U)), U ~ Uniform(0,1)
    gumbels = (
        -torch.empty_like(logits, memory_format=torch.legacy_contiguous_format)
        .exponential_()
        .log()
    )  # ~ Gumbel(0, 1)
    
    # MPS'deki sayısal kararlılık sorunu için clamp ekle (bug fix)
    gumbels = torch.clamp(gumbels, -30, 30)
    
    # Logitlere Gumbel gürültüsü ekle, temperature ile ölçekle
    gumbels = (logits + gumbels) / tau
    y_soft  = gumbels.softmax(dim)  # Yumuşak Gumbel-Softmax

    if hard:
        # Straight-Through Estimator (STE):
        # İleri geçiş: en yüksek değeri 1, diğerlerini 0 yapan one-hot vektör
        # Geri geçiş: yumuşak Gumbel-Softmax'ın gradient'i kullanılır
        # Bu sayede kesikli seçim yapılır ama gradient geçer!
        index  = y_soft.max(dim, keepdim=True)[1]  # En yüksek değerin indeksi
        y_hard = torch.zeros_like(
            logits, memory_format=torch.legacy_contiguous_format
        ).scatter_(dim, index, 1.0)
        # y_hard - y_soft.detach() + y_soft:
        # İleri: y_hard (one-hot), Geri: y_soft gradient'i
        ret = y_hard - y_soft.detach() + y_soft
    else:
        ret = y_soft  # Yumuşak versiyon (reparameterization trick)
    
    return ret

In [ ]:
DiscreteVAEOutput = namedtuple("DiscreteVAEOutput",
                               ["output", "logits", "codings_prob"])

class DiscreteVAE(nn.Module):
    """
    Discrete VAE: Her latent değişken k kategoriden birini seçer.
    
    Latent space: coding_length adet değişken, her biri n_codes kategorisinden biri
    Toplam kombinasyon: n_codes ^ coding_length
    """
    def __init__(self, coding_length=32, n_codes=16, temperature=1.0):
        super().__init__()
        self.coding_length = coding_length  # Kaç adet kesikli değişken?
        self.n_codes       = n_codes        # Her değişken kaç kategoriden oluşuyor?
        self.temperature   = temperature    # Gumbel-Softmax temperature
        
        # ── ENCODER ────────────────────────────────────────────────────────
        # Çıkış: [B, coding_length, n_codes] — her değişken için n_codes logit
        self.encoder = nn.Sequential(
            nn.Flatten(),
            nn.Linear(1 * 28 * 28, 256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, coding_length * n_codes),  # coding_length*n_codes = 32*16 = 512
            nn.Unflatten(dim=1, unflattened_size=(coding_length, n_codes))  # [B, 32, 16]
        )
        
        # ── DECODER ────────────────────────────────────────────────────────
        # Giriş: Gumbel-Softmax çıktısı [B, coding_length, n_codes] → flatten → [B, 512]
        self.decoder = nn.Sequential(
            nn.Flatten(),
            nn.Linear(coding_length * n_codes, 128), nn.ReLU(),
            nn.Linear(128, 256), nn.ReLU(),
            nn.Linear(256, 1 * 28 * 28), nn.Sigmoid(),
            nn.Unflatten(dim=1, unflattened_size=(1, 28, 28))
        )

    def forward(self, X):
        logits = self.encoder(X)   # [B, 32, 16] — ham logitler
        # Gumbel-Softmax: hard=True → one-hot forward, soft backward (STE)
        codings_prob = gumbel_softmax(logits, tau=self.temperature, hard=True)
        output = self.decoder(codings_prob)
        return DiscreteVAEOutput(output, logits, codings_prob)

In [ ]:
def d_vae_loss(y_pred, y_target, kl_weight=1.0):
    """
    Discrete VAE kayıp fonksiyonu.
    
    KL Divergence (Kategorik Dağılım):
    KL(q(z|x) || p(z)) burada p(z) = Uniform (tüm kategoriler eşit olasılıklı)
    
    KL = Σ q(z) * (log q(z) - log(1/k)) = Σ q(z) * (log q(z) + log k)
    """
    output, logits, _ = y_pred
    
    # Logitlerden olasılık dağılımı hesapla
    codings_prob = F.softmax(logits, -1)  # [B, coding_length, n_codes]
    
    # k: kategori sayısı (n_codes = 16)
    k = logits.new_tensor(logits.size(-1))  # logits ile aynı device ve dtype
    
    # KL = Σ q(z) * (log q(z) + log k) = q * log(q*k)
    # Her örnek için coding_length boyutu ve n_codes boyutu üzerinde topla
    kl_div = (codings_prob * (codings_prob.log() + k.log())).sum(dim=(1, 2))
    
    return F.mse_loss(output, y_target) + kl_weight * kl_div.mean() / 784

In [ ]:
# Temperature Annealing: Eğitim boyunca temperature'ı kademeli düşür
# Başlangıç: τ=1 (yumuşak, keşif) → Bitiş: τ=0.1 (sert, kesikli)
n_epochs = 20

def annealing(model, epoch):
    """
    Her epoch başında çağrılır ve temperature'ı azaltır.
    epoch=0 → temperature=1.0
    epoch=19 → temperature=0.1
    """
    model.temperature = 1 - 0.9 * epoch / n_epochs

torch.manual_seed(42)
d_vae = DiscreteVAE().to(device)
optimizer = torch.optim.NAdam(d_vae.parameters(), lr=0.001)

# epoch_callback=annealing: Her epoch başında temperature'ı güncelle
history = train(d_vae, optimizer, d_vae_loss, rmse, train_loader, valid_loader,
                n_epochs=n_epochs, epoch_callback=annealing)

In [ ]:
# Discrete VAE ile yeni görüntüler üret
torch.manual_seed(42)
n_images = 3 * 7

# Rastgele kesikli kodlar üret: her değişken için 0 ile k-1 arası tamsayı
codings = torch.randint(0, d_vae.n_codes,           # [0, 15] aralığında
                        (n_images, d_vae.coding_length),  # [21, 32] boyutu
                        device=device)

# Tamsayı kodları one-hot vektörlere dönüştür
# [21, 32] → [21, 32, 16] — her değişken için 16 boyutlu one-hot
codings_prob = F.one_hot(codings, num_classes=d_vae.n_codes).float()

with torch.no_grad():
    images = d_vae.decoder(codings_prob)

In [ ]:
# Üretilen görüntüleri göster
plot_multiple_images(images, 7)
plt.show()

# 9. Generative Adversarial Networks (GANs — Üretici Çekişmeli Ağlar)

GAN, iki ağın birbirini geliştirerek gerçekçi veri üretmeyi öğrenen bir çerçevedir.

## Mimari

```
z ~ N(0,1) → [Generator G] → Sahte görüntü
                                      ↓
Gerçek görüntü ─────────────────→ [Discriminator D] → Gerçek mi? (0/1)
```

- **Generator (G)**: Rastgele gürültüden gerçekçi görüntüler üretmeye çalışır
- **Discriminator (D)**: Gerçek ve sahte görüntüleri ayırt etmeye çalışır

## Eğitim Süreci (Mini-Max Oyunu)

**Discriminator Adımı**:
1. Gerçek görüntüler için kayıp: `BCE(D(x_real), 1)` → D gerçeği 1 yapmalı
2. Sahte görüntüler için kayıp: `BCE(D(G(z)), 0)` → D sahteyı 0 yapmalı
3. Toplam kayıp: `real_loss + fake_loss`

**Generator Adımı**:
1. Discriminator'ı kandırmak: `BCE(D(G(z)), 1)` → G, D'nin sahteyı 1 görmesini ister
2. D'nin gradientleri dondurulur (sadece G güncellenir)


In [ ]:
torch.manual_seed(42)

codings_dim = 32  # Latent vektör boyutu

# ── GENERATOR ────────────────────────────────────────────────────────────
# Giriş: z ~ N(0,1) boyutu [B, 32]
# Çıkış: Sahte görüntü [B, 1, 28, 28]
generator = nn.Sequential(
    nn.Linear(codings_dim, 128), nn.ReLU(),
    nn.Linear(128, 256), nn.ReLU(),
    nn.Linear(256, 1 * 28 * 28), nn.Sigmoid(),  # [0,1] aralığı
    nn.Unflatten(dim=1, unflattened_size=(1, 28, 28))
).to(device)

# ── DISCRIMINATOR ─────────────────────────────────────────────────────────
# Giriş: Görüntü [B, 1, 28, 28]
# Çıkış: Gerçek olasılığı [B, 1], Sigmoid → [0,1]
discriminator = nn.Sequential(
    nn.Flatten(),
    nn.Linear(1 * 28 * 28, 256), nn.ReLU(),
    nn.Linear(256, 128), nn.ReLU(),
    nn.Linear(128, 1), nn.Sigmoid()  # 1 = gerçek, 0 = sahte
).to(device)

In [ ]:
def train_gan(generator, discriminator, train_loader, codings_dim, n_epochs=20,
              g_lr=1e-3, d_lr=5e-4):
    """
    GAN eğitim döngüsü. Her batch'te önce Discriminator, sonra Generator güncellenir.
    
    Args:
        generator: Generator modeli
        discriminator: Discriminator modeli
        train_loader: Eğitim DataLoader'ı
        codings_dim: Latent vektör boyutu
        n_epochs: Epoch sayısı
        g_lr: Generator learning rate
        d_lr: Discriminator learning rate (genellikle G'den küçük)
    """
    criterion = nn.BCELoss()  # Binary Cross-Entropy: gerçek/sahte sınıflandırma
    
    # Ayrı optimizer'lar: G ve D bağımsız olarak güncellenir
    generator_opt     = torch.optim.NAdam(generator.parameters(),     lr=g_lr)
    discriminator_opt = torch.optim.NAdam(discriminator.parameters(), lr=d_lr)
    
    for epoch in range(n_epochs):
        print(f"Epoch {epoch + 1}/{n_epochs}", end="")
        
        for real_images, _ in train_loader:
            real_images = real_images.to(device)
            batch_size  = real_images.size(0)
            
            # ════════════════════════════════════════════════════════════════
            # ADIM 1: DISCRİMİNATOR EĞİTİMİ
            # ════════════════════════════════════════════════════════════════
            
            # 1a. Gerçek görüntüler için: D(x_real) ≈ 1 olmalı
            pred_real = discriminator(real_images)
            ones      = torch.ones(batch_size, 1, device=device)   # Hedef: 1 (gerçek)
            real_loss = criterion(pred_real, ones)
            
            # 1b. Sahte görüntüler için: D(G(z)) ≈ 0 olmalı
            codings    = torch.randn(batch_size, codings_dim, device=device)
            fake_images = generator(codings).detach()  # .detach(): G'nin gradient'ini engelle
            pred_fake   = discriminator(fake_images)
            zeros       = torch.zeros(batch_size, 1, device=device)  # Hedef: 0 (sahte)
            fake_loss   = criterion(pred_fake, zeros)
            
            # Discriminator total loss = gerçek kaybı + sahte kaybı
            discriminator_loss = real_loss + fake_loss
            discriminator_opt.zero_grad()
            discriminator_loss.backward()
            discriminator_opt.step()
            
            # ════════════════════════════════════════════════════════════════
            # ADIM 2: GENERATOR EĞİTİMİ
            # ════════════════════════════════════════════════════════════════
            
            codings     = torch.randn(batch_size, codings_dim, device=device)
            fake_images = generator(codings)  # Bu sefer .detach() YOK (G güncelleniyor)
            
            # D'nin parametrelerini dondur (sadece G güncelleniyor)
            for p in discriminator.parameters():
                p.requires_grad = False
            
            # G, D'yi kandırmak istiyor: D(G(z)) ≈ 1 olsun
            pred_fake      = discriminator(fake_images)
            generator_loss = criterion(pred_fake, ones)   # Hedef: 1 (kandır!)
            generator_opt.zero_grad()
            generator_loss.backward()
            generator_opt.step()
            
            # D'nin parametrelerini tekrar aç
            for p in discriminator.parameters():
                p.requires_grad = True
        
        print(f" | discriminator loss: {discriminator_loss.item():.4f}", end="")
        print(f" | generator loss: {generator_loss.item():.4f}")
        
        # Her 10 epoch'ta bir ve son epoch'ta üretilen görüntüleri göster
        if epoch % 10 == 0 or epoch == n_epochs - 1:
            plot_multiple_images(fake_images.detach(), 8)
            plt.show()

In [ ]:
# GAN eğitimi: seed 41 kullanıyoruz
# GAN eğitimi moda (seed, lr vb.) çok duyarlıdır; farklı seed'ler çok farklı sonuçlar verir
torch.manual_seed(41)  # Farklı seed deneyerek daha iyi sonuç alınabiliyor
train_gan(generator, discriminator, train_loader, codings_dim)

In [ ]:
# Eğitilmiş Generator ile yeni görüntüler üret
torch.manual_seed(42)
n_images = 3 * 7
generator.eval()

# Latent vektörler örnekle
codings = torch.randn(n_images, codings_dim, device=device)

with torch.no_grad():
    generated_images = generator(codings)

In [ ]:
# Üretilen görüntüleri göster
plot_multiple_images(generated_images, 7)
plt.show()

# 10. Deep Convolutional GAN (DCGAN — Derin Evrişimli Çekişmeli Ağ)

**DCGAN**, tam bağlı katmanlar yerine **convolutional** katmanlar kullanan geliştirilmiş bir GAN mimarisidir.

## DCGAN'ın Temel Öğeleri

| Özellik | Açıklama |
|---------|----------|
| **Batch Normalization** | Her katmandan sonra, eğitimi stabilize eder |
| **ConvTranspose** (Generator) | Görüntüyü kademeli büyütür |
| **Strided Convolution** (Discriminator) | MaxPool yerine stride ile küçültme |
| **Dropout** | Discriminator'da regularizasyon |
| **Leaky ReLU** | Standart implementasyonda, burada ReLU kullanılmış |

## Generator Akışı
`z[100] → 128*7*7 → [128,7,7] → [64,14,14] → [1,28,28]`

## Discriminator Akışı
`[1,28,28] → [32,14,14] → [64,7,7] → [3136] → [1]`


In [ ]:
torch.manual_seed(1)

dc_codings_dim = 100  # DCGAN için daha büyük latent boyut

# ── DCGAN GENERATOR ───────────────────────────────────────────────────────
dc_generator = nn.Sequential(
    # 100 boyutlu latent vektör → 128*7*7 = 6272 nöron
    nn.Linear(dc_codings_dim, 128 * 7 * 7),
    
    # 1D → 3D: [B, 6272] → [B, 128, 7, 7]
    nn.Unflatten(dim=1, unflattened_size=(128, 7, 7)),
    
    # BatchNorm: eğitimi stabilize eder, internal covariate shift azaltır
    nn.BatchNorm2d(128),
    
    # ConvTranspose: [B, 128, 7, 7] → [B, 64, 14, 14] (stride=2 ile büyüt)
    nn.ConvTranspose2d(128, 64, kernel_size=3, stride=2, padding=1,
                       output_padding=1), nn.ReLU(),
    nn.BatchNorm2d(64),
    
    # ConvTranspose: [B, 64, 14, 14] → [B, 1, 28, 28] (stride=2 ile büyüt)
    nn.ConvTranspose2d(64, 1, kernel_size=3, stride=2, padding=1,
                       output_padding=1), nn.Sigmoid()
).to(device)

# ── DCGAN DISCRIMINATOR ───────────────────────────────────────────────────
dc_discriminator = nn.Sequential(
    # Strided Conv: [B, 1, 28, 28] → [B, 32, 14, 14] (stride=2 ile küçült)
    # MaxPool yerine strided conv: daha iyi özellik öğrenimi
    nn.Conv2d(1, 32, kernel_size=5, stride=2, padding=2), nn.ReLU(),
    nn.Dropout(0.4),   # %40 dropout: overfitting önleme
    
    # Strided Conv: [B, 32, 14, 14] → [B, 64, 7, 7]
    nn.Conv2d(32, 64, kernel_size=5, stride=2, padding=2), nn.ReLU(),
    nn.Dropout(0.4),
    
    nn.Flatten(),                      # [B, 64, 7, 7] → [B, 3136]
    nn.Linear(64 * 7 * 7, 1), nn.Sigmoid()  # [B, 3136] → [B, 1]
).to(device)

In [ ]:
# DCGAN'ı eğit — aynı train_gan fonksiyonunu kullanıyoruz
torch.manual_seed(42)
train_gan(dc_generator, dc_discriminator, train_loader, dc_codings_dim)

In [ ]:
# DCGAN ile yeni görüntüler üret
torch.manual_seed(42)
with torch.no_grad():
    codings = torch.randn(8, dc_codings_dim, device=device)
    generated_images = dc_generator(codings)

plot_multiple_images(generated_images, 8)

# 11. Diffusion Models (Yayınım Modelleri)

Diffusion modeller, modern görüntü üretiminin temelini oluşturan güçlü üretici modellerdir (Stable Diffusion, DALL-E 3, Midjourney).

## Temel Fikir

**İleri Süreç (Forward Diffusion)**: Gerçek görüntüye adım adım Gaussian gürültü ekle  
**Geri Süreç (Reverse Diffusion)**: Gürültüden orijinal görüntüyü geri kurtarmayı öğren

```
x₀ (gerçek görüntü)
  → x₁ (biraz gürültülü)
  → x₂ (daha gürültülü)
  → ...
  → x_T (saf Gaussian gürültü)  ← Burada başlıyoruz

Geri süreç:
x_T → x_{T-1} → ... → x₁ → x₀ (üretilen görüntü)
```

## Variance Schedule

Her adımda ne kadar gürültü ekleneceğini belirleyen `βₜ` parametreleri.


In [ ]:
def variance_schedule(T, s=0.008, max_beta=0.999):
    """
    Cosine variance schedule (Improved DDPM, Nichol & Dhariwal 2021).
    
    Doğrusal schedule yerine cosine schedule kullanmak, başlangıç ve
    bitişteki ani değişimleri yumuşatarak daha iyi sonuçlar verir.
    
    Args:
        T: Toplam diffusion adım sayısı
        s: Küçük sabit (cosine'in çok küçük başlamaması için)
        max_beta: Beta'nın maksimum değeri (sayısal kararlılık için sınır)
    
    Returns:
        alphas: 1 - beta değerleri [T+1]
        betas: Gürültü ekleme oranları [T+1], beta[0]=0 (padding)
        alpha_bars: Kümülatif çarpım ∏αₜ [T+1]
    """
    t = torch.linspace(0, T, T + 1)  # 0'dan T'ye T+1 eşit adım
    
    # Cosine schedule: f(t) = cos((t/T + s)/(1+s) * π/2)²
    f = torch.cos((t / T + s) / (1 + s) * torch.pi / 2) ** 2
    
    # alpha_bar: f(t)/f(0) ile normalize et
    # alpha_bar[0] = 1 (gürültü yok), alpha_bar[T] ≈ 0 (saf gürültü)
    alpha_bars = f / f[0]
    
    # beta: ardışık alpha_bar oranından hesapla
    # beta[t] = 1 - alpha_bar[t] / alpha_bar[t-1]
    betas = (1 - (f[1:] / f[:-1])).clamp(max=max_beta)
    betas = torch.cat([torch.zeros(1), betas])  # beta[0]=0 (indeksleme kolaylığı için)
    
    alphas = 1 - betas  # alpha[t] = 1 - beta[t]
    
    return alphas, betas, alpha_bars

torch.manual_seed(42)
T = 4000  # Toplam adım sayısı (Improved DDPM: 4000 adım)
alphas, betas, alpha_bars = variance_schedule(T)

In [ ]:
# alpha_bars ve betas'ı görselleştir
plt.figure(figsize=(6, 3))
plt.plot(betas,       "r--", label=r"$\beta_t$ (gürültü ekleme oranı)")
plt.plot(alpha_bars,  "b",   label=r"$\bar{\alpha}_t$ (kalan sinyal oranı)")
plt.axis([0, T, 0, 1.01])
plt.grid(True)
plt.xlabel("$t$ (zaman adımı)")
plt.ylabel(r"Değer")
plt.legend()
plt.show()
# beta artar: Her adımda daha fazla gürültü
# alpha_bar azalır: Sinyal giderek kaybolur, t=T'de saf gürültü

## 11.1 Forward Diffusion (İleri Yayınım)

DDPM makalesinin 4. denklemi ile, herhangi bir t adımındaki gürültülü görüntüyü **tek adımda** hesaplayabiliriz:

```
x_t = √ᾱₜ * x₀ + √(1-ᾱₜ) * ε,   ε ~ N(0,I)
```

Bu formül, t adıma kadar tüm gürültü ekleme işlemini tek seferde yapar.


In [ ]:
def forward_diffusion(x0, t):
    """
    DDPM Denklem (4): x₀'dan x_t'yi tek adımda hesapla.
    
    x_t = √ᾱₜ * x₀ + √(1-ᾱₜ) * ε
    
    - √ᾱₜ * x₀: Orijinal görüntünün ᾱₜ oranındaki katkısı
    - √(1-ᾱₜ) * ε: Gürültünün (1-ᾱₜ) oranındaki katkısı
    
    Args:
        x0: Orijinal temiz görüntü [B, C, H, W]
        t: Zaman adımı indeksleri [B, 1]
    
    Returns:
        xt: t adımındaki gürültülü görüntü
        eps: Eklenen gürültü (modelin tahmin etmeye çalışacağı hedef)
    """
    eps = torch.randn_like(x0)  # ε ~ N(0, I) — standart Gaussian gürültü
    
    # alpha_bars[t]: t adımındaki kümülatif sinyal oranı
    # Boyut uyumu için: alpha_bars[t] skaler → x0 ile çarpılabilmesi için yayma gerekli
    xt = alpha_bars[t].sqrt() * x0 + (1 - alpha_bars[t]).sqrt() * eps
    
    return xt, eps  # xt: gürültülü görüntü, eps: modelin tahmin edeceği gürültü

In [ ]:
class DiffusionSample(namedtuple("DiffusionSampleBase", ["xt", "t"])):
    """
    Diffusion modelinin bir batch girdisini temsil eden named tuple.
    
    Fields:
        xt: t adımındaki gürültülü görüntü tensoru [B, C, H, W]
        t:  Zaman adımı indeksleri [B, 1]
    """
    def to(self, device):
        """
        Named tuple'ın tüm tensor'larını belirtilen device'a taşır.
        Bu metot, train() fonksiyonunda X_batch.to(device) çağrısıyla uyumlu olması için gerekli.
        """
        return DiffusionSample(self.xt.to(device), self.t.to(device))

## 11.2 DiffusionDataset — Eğitim Verisi Hazırlama

Diffusion modeli şunu öğrenir: `x_t` görüntüsü ve `t` zaman adımı verildiğinde, `x_t`'ye hangi `ε` gürültüsünün eklendiğini tahmin et.

In [ ]:
class DiffusionDataset:
    """
    Fashion MNIST veri setini diffusion eğitimi için hazırlar.
    
    Her örnek için:
    1. Görüntüyü [-1, +1] aralığına ölçekle (diffusion için standart aralık)
    2. Rastgele bir t zaman adımı seç
    3. Forward diffusion ile gürültülü x_t üret
    4. (DiffusionSample(xt, t), ε) çifti döndür
    
    Model bu çiftten ε'yu (gürültüyü) tahmin etmeyi öğrenecek.
    """
    def __init__(self, dataset):
        self.dataset = dataset

    def __getitem__(self, i):
        x0, _ = self.dataset[i]    # Görüntüyü al, etiketi yoksay
        x0 = (x0 * 2) - 1         # [0,1] → [-1,+1] aralığına ölçekle
        
        # Rastgele t seç: [1, T] aralığında tekdüze örnekleme
        t = torch.randint(1, T + 1, size=[1])
        
        # t adımındaki gürültülü görüntü ve hedef gürültüyü üret
        xt, eps = forward_diffusion(x0, t)
        
        return DiffusionSample(xt, t), eps  # (giriş, hedef) çifti

    def __len__(self):
        return len(self.dataset)

# Fashion MNIST'i DiffusionDataset ile sar
train_set    = DiffusionDataset(train_data)
train_loader = DataLoader(train_set, batch_size=32, shuffle=True)

In [ ]:
# Validation seti için DataLoader
valid_set    = DiffusionDataset(valid_data)
valid_loader = DataLoader(valid_set, batch_size=32)

In [ ]:
# Hızlı kontrol: verinin doğru oluşturulduğunu doğrula

def original_image(sample, noise):
    """
    x_t ve ε verildiğinde orijinal x₀'ı geri hesaplar.
    x₀ = (x_t - √(1-ᾱₜ) * ε) / √ᾱₜ
    """
    alpha_bars_t = torch.gather(alpha_bars, dim=0, index=sample.t.squeeze(1))
    alpha_bars_t = alpha_bars_t.view(-1, 1, 1, 1)  # Yayma için boyut ekle
    x0 = (sample.xt - (1 - alpha_bars_t).sqrt() * noise) / alpha_bars_t.sqrt()
    return torch.clamp((x0 + 1) / 2, 0, 1)  # [-1,1] → [0,1]

torch.manual_seed(42)
sample, eps = next(iter(train_loader))  # İlk batch'i al
x0 = original_image(sample, eps).to(device)

print("Orijinal görüntüler (geri hesaplandı)")
plot_multiple_images(x0[:8])
plt.show()

print("Zaman adımları:", sample.t[:8].view(-1).tolist())

print("Gürültülü görüntüler (x_t)")
plot_multiple_images(sample.xt[:8])
plt.show()

print("Hedef gürültü (ε — modelin tahmin edeceği)")
plot_multiple_images(eps[:8])
plt.show()

## 11.3 Time Encoding (Zaman Kodlaması)

Diffusion modeli her adım `t` için farklı davranmalıdır. Bunu sağlamak için zaman adımını **sinuzoidal kodlama** ile sürekli bir vektöre dönüştürüyoruz.

Bu, Transformer'lardaki **pozisyonel kodlama** ile aynı tekniği kullanır:
```
PE(t, 2i)   = sin(t / 10000^(2i/d))
PE(t, 2i+1) = cos(t / 10000^(2i/d))
```


In [ ]:
embed_dim = 64  # Zaman kodlaması boyutu

class TimeEncoding(nn.Module):
    """
    Transformer pozisyonel kodlamasına benzer sinuzoidal zaman kodlaması.
    
    Her t zaman adımını embed_dim boyutlu bir vektöre dönüştürür.
    Çift indeksler → sin, Tek indeksler → cos
    
    Neden sinuzoidal?
    - Sürekli ve periyodik: Yakın t değerleri benzer kodlamalar alır
    - Farklı frekanslar: Her boyut farklı bir zamansal ölçek yakalar
    - Öğrenilmez: Sabit, register_buffer olarak saklanır
    """
    def __init__(self, T, embed_dim):
        super().__init__()
        assert embed_dim % 2 == 0, "embed_dim çift sayı olmalı"
        
        # p: [T+1, 1] — tüm zaman adımları
        p = torch.arange(T + 1).unsqueeze(1)
        
        # Açı = t / (10000 ^ (2i/d))
        angle = p / 10_000 ** (torch.arange(0, embed_dim, 2) / embed_dim)
        
        # Sinuzoidal zaman kodlaması matrisi: [T+1, embed_dim]
        te = torch.empty(T + 1, embed_dim)
        te[:, 0::2] = torch.sin(angle)  # Çift indeksler: sin
        te[:, 1::2] = torch.cos(angle)  # Tek indeksler: cos
        
        # register_buffer: model parametresi değil ama state_dict'e dahil edilir
        # device değiştiğinde (GPU/CPU) otomatik taşınır
        self.register_buffer("time_encodings", te)

    def forward(self, t):
        """t zaman adımlarına karşılık gelen zaman kodlamalarını döndürür."""
        return self.time_encodings[t]

## 11.4 UNet Benzeri Diffusion Modeli

Diffusion modeli için **UNet** mimarisi kullanıyoruz. UNet aşağı yolda (downsampling) özellik çıkarır, yukarı yolda (upsampling) detayları geri getirir.

**Skip Connections**: Aşağı yoldan yukarı yola doğrudan bağlantılar, ince detayların korunmasını sağlar.

**Separable Convolution**: Depthwise + Pointwise conv kombinasyonu, standart conv'a göre çok daha az parametre kullanır.


In [ ]:
class SeparableConv2d(nn.Module):
    """
    Separable (Ayrılabilir) Convolution: Depthwise + Pointwise.
    
    Standart Conv2d'ye göre çok daha az parametre:
    - Depthwise conv: Her kanal bağımsız filtrelenir (in_ch * k * k parametre)
    - Pointwise conv: 1x1 conv ile kanallar karıştırılır (in_ch * out_ch parametre)
    
    MobileNet gibi hafif mimarilerde yaygın kullanılır.
    """
    def __init__(self, in_channels, out_channels, kernel_size, padding):
        super().__init__()
        # Depthwise: Her girdi kanalı kendi filtresiyle konvolüsyon yapar
        # groups=in_channels: Her kanal bağımsız işlenir
        self.depthwise = nn.Conv2d(in_channels, in_channels, kernel_size,
                                   padding=padding, groups=in_channels)
        # Pointwise: 1×1 conv ile kanalları karıştır ve out_channels'a dönüştür
        self.pointwise = nn.Conv2d(in_channels, out_channels, kernel_size=1)

    def forward(self, X):
        return self.pointwise(self.depthwise(X))


class DiffusionModel(nn.Module):
    """
    UNet benzeri diffusion modeli.
    
    Giriş:
    - xt: t adımındaki gürültülü görüntü [B, 1, 28, 28]
    - t: Zaman adımı indeksi [B, 1]
    
    Çıkış:
    - ε̂: Tahmin edilen gürültü [B, 1, 28, 28] (hedef: gerçek ε)
    
    Mimari:
    - Init: 1 → 16 kanal
    - Down path: 16 → 32 → 64 → 128 (MaxPool ile downsample)
    - Up path: 128 → 64 → 32 → 16 (Interpolate ile upsample)
    - Skip connections: Her down bloktan up bloka cross-skip
    - Time adapter: Her blokta zaman bilgisini feature map'e ekle
    """
    def __init__(self, T=T, embed_dim=64):
        super().__init__()
        self.time_encoding = TimeEncoding(T, embed_dim)

        # ── BAŞLANGIÇ BLOĞU ───────────────────────────────────────────────
        dim = 16
        # ConstantPad: görüntü kenarlıklarını sıfırla doldurup
        # sonraki konvolüsyonlarda boyut uyumunu sağlar
        self.pad           = nn.ConstantPad2d((3, 3, 3, 3), 0)  # [1,28,28] → [1,34,34]
        self.init_conv     = nn.Conv2d(1, dim, kernel_size=3)    # [1,34,34] → [16,32,32]
        self.init_bn       = nn.BatchNorm2d(dim)
        self.time_adapter_init = nn.Linear(embed_dim, dim)       # Zaman kodunu boyuta uyarla

        # ── AŞAĞI YOL (DOWNSAMPLING PATH) ────────────────────────────────
        # Her blok: SeparableConv × 2, BatchNorm, ardından MaxPool
        # 16 → 32 → 64 → 128 kanal (her adımda boyut yarıya iner)
        self.down_blocks         = nn.ModuleList()
        self.skip_convs          = nn.ModuleList()
        self.time_adapters_down  = nn.ModuleList()
        
        in_dim = dim
        for dim in (32, 64, 128):
            block = nn.Sequential(
                nn.ReLU(),
                SeparableConv2d(in_dim, dim, kernel_size=3, padding=1),
                nn.BatchNorm2d(dim),
                nn.ReLU(),
                SeparableConv2d(dim, dim, kernel_size=3, padding=1),
                nn.BatchNorm2d(dim)
            )
            # Skip connection: Residual bağlantı için kanal sayısını ayarla
            # stride=2: Feature map boyutunu MaxPool ile eşleştirmek için
            skip_conv = nn.Conv2d(in_dim, dim, kernel_size=1, stride=2)
            
            self.down_blocks.append(block)
            self.skip_convs.append(skip_conv)
            self.time_adapters_down.append(nn.Linear(embed_dim, dim))
            in_dim = dim

        # ── YUKARI YOL (UPSAMPLING PATH) ──────────────────────────────────
        # Her blok: ConvTranspose × 2, BatchNorm, ardından Interpolate + Cross-skip
        # 128 → 64 → 32 → 16 kanal (her adımda boyut ikiye çıkar)
        self.up_blocks        = nn.ModuleList()
        self.skip_up_convs    = nn.ModuleList()
        self.time_adapters_up = nn.ModuleList()
        
        for dim in (64, 32, 16):
            block = nn.Sequential(
                nn.ReLU(),
                nn.ConvTranspose2d(in_dim, dim, 3, padding=1),
                nn.BatchNorm2d(dim),
                nn.ReLU(),
                nn.ConvTranspose2d(dim, dim, 3, padding=1),
                nn.BatchNorm2d(dim)
            )
            skip_conv = nn.Conv2d(in_dim, dim, kernel_size=1)
            
            self.up_blocks.append(block)
            self.skip_up_convs.append(skip_conv)
            self.time_adapters_up.append(nn.Linear(embed_dim, dim))
            
            # Sonraki bloğun giriş boyutu = mevcut çıkış + cross-skip (dim * 3 değil dikkat)
            in_dim = dim * 3  # Çünkü torch.cat ile cross-skip birleştiriliyor

        # Son konvolüsyon: Gürültü tahminini 1 kanala düşür
        self.final_conv = nn.Conv2d(in_dim, 1, 3, padding=1)

    def forward(self, sample):
        # ── ZAMAN KODLAMASI ───────────────────────────────────────────────
        # sample.t: [B, 1] → squeeze → [B] → time encoding → [B, embed_dim]
        time_enc = self.time_encoding(sample.t.squeeze(1))
        
        # ── BAŞLANGIÇ ────────────────────────────────────────────────────
        z = self.pad(sample.xt)            # [B, 1, 28, 28] → [B, 1, 34, 34]
        z = F.relu(self.init_bn(self.init_conv(z)))  # → [B, 16, 32, 32]
        
        # Zaman bilgisini feature map'e ekle (broadcast ile spatial boyutlara yay)
        z = z + self.time_adapter_init(time_enc)[:, :, None, None]
        skip = z
        cross_skips = []  # Down bloklardan yukarı yola aktarılacak özellikler

        # ── AŞAĞI YOL ────────────────────────────────────────────────────
        for block, skip_conv, time_adapter in zip(
                self.down_blocks, self.skip_convs, self.time_adapters_down):
            z = block(z)                   # SeparableConv bloğu uygula
            cross_skips.append(z)          # Cross-skip için sakla
            z = F.max_pool2d(z, 3, stride=2, padding=1)  # Boyutu yarıya indir
            skip_link = skip_conv(skip)    # Residual bağlantı (stride=2 ile küçültülmüş)
            z = z + skip_link              # Residual ekle
            z = z + time_adapter(time_enc)[:, :, None, None]  # Zaman bilgisi ekle
            skip = z

        # ── YUKARI YOL ───────────────────────────────────────────────────
        for block, skip_up_conv, time_adapter in zip(
                self.up_blocks, self.skip_up_convs, self.time_adapters_up):
            z = block(z)                   # ConvTranspose bloğu uygula
            z = F.interpolate(z, scale_factor=2, mode="nearest")  # Boyutu 2x büyüt
            
            # Skip bağlantısını da büyüt
            skip_link = F.interpolate(skip, scale_factor=2, mode="nearest")
            skip_link = skip_up_conv(skip_link)
            
            z = z + skip_link              # Residual bağlantı
            z = z + time_adapter(time_enc)[:, :, None, None]  # Zaman bilgisi
            
            # Cross-skip: Aşağı yoldan gelen özellikleri birleştir (UNet karakteristiği)
            cross_skip = cross_skips.pop()  # Lifo: son eklenen ilk alınır
            z = torch.cat([z, cross_skip], dim=1)  # Kanal boyutunda birleştir
            skip = z

        # ── SON KATMAN ────────────────────────────────────────────────────
        out = self.final_conv(z)     # [B, in_dim, H, W] → [B, 1, H, W]
        out = out[:, :, 2:-2, 2:-2]  # Padding'den kaynaklanan kenar pikselleri kes
        return out.contiguous()      # Bellek düzenini düzelt

In [ ]:
# Diffusion modelini eğit
torch.manual_seed(42)
diffusion_model = DiffusionModel().to(device)

# HuberLoss: MSE ve MAE'nin kombinasyonu — büyük hatalara karşı daha sağlam
huber = nn.HuberLoss()

optimizer = torch.optim.NAdam(diffusion_model.parameters(), lr=3e-3)
rmse = torchmetrics.MeanSquaredError(squared=False).to(device)

history = train(diffusion_model, optimizer, huber, rmse, train_loader,
                valid_loader, n_epochs=20)

## 11.5 DDPM ile Görüntü Üretimi

Model eğitildikten sonra **geri diffusion süreciyle** yeni görüntüler üretiyoruz.

DDPM geri geçişi (t'den t-1'e):
```
x_{t-1} = (1/√αₜ) * (xₜ - βₜ/√(1-ᾱₜ) * ε̂(xₜ,t)) + √(1-αₜ) * z
```
- `ε̂(xₜ,t)`: Modelin tahmin ettiği gürültü
- `z ~ N(0,I)`: Yeni eklenen Gaussian gürültü (t>1 için)


In [ ]:
def generate_ddpm(model, batch_size=32):
    """
    DDPM (Denoising Diffusion Probabilistic Models) ile görüntü üretimi.
    
    T adımda geri diffusion: x_T → x_{T-1} → ... → x_1 → x_0
    Her adımda model gürültüyü tahmin eder ve çıkarır.
    
    DDPM dezavantajı: T=4000 adım gerektirir → çok yavaş üretim
    """
    model.eval()
    with torch.no_grad():
        # Saf Gaussian gürültüden başla: x_T ~ N(0, I)
        xt = torch.randn([batch_size, 1, 28, 28], device=device)
        
        # T'den 1'e geri git (T, T-1, T-2, ..., 1)
        for t in range(T, 0, -1):
            print(f"\rt = {t}", end=" ")  # İlerlemeyi göster
            
            alpha_t     = alphas[t]      # αₜ
            beta_t      = betas[t]       # βₜ
            alpha_bar_t = alpha_bars[t]  # ᾱₜ
            
            # t>1: Yeni gürültü ekle; t=1: Gürültü ekleme (son adım)
            noise = (torch.randn(xt.shape, device=device)
                     if t > 1 else torch.zeros(xt.shape, device=device))
            
            # Modele mevcut gürültülü görüntüyü ve zaman adımını ver
            t_batch   = torch.full((batch_size, 1), t, device=device)
            sample    = DiffusionSample(xt, t_batch)
            eps_pred  = model(sample)  # ε̂: tahmin edilen gürültü
            
            # DDPM geri geçiş formülü:
            # x_{t-1} = (1/√αₜ) * (xₜ - βₜ/√(1-ᾱₜ) * ε̂) + √(1-αₜ) * z
            xt = (1 / alpha_t.sqrt()
                  * (xt - beta_t / (1 - alpha_bar_t).sqrt() * eps_pred)
                  + (1 - alpha_t).sqrt() * noise)
        
        # [-1,1] → [0,1] aralığına dönüştür ve geçerli piksel aralığında tut
        return torch.clamp((xt + 1) / 2, 0, 1)

torch.manual_seed(42)
X_gen = generate_ddpm(diffusion_model)  # 32 görüntü üret (YAVAŞ: 4000 adım!)

In [ ]:
# Üretilen görüntüleri göster
plot_multiple_images(X_gen, 8)
plt.show()

## 11.6 DDIM ile Hızlı Görüntü Üretimi

**DDIM** (Denoising Diffusion Implicit Models), DDPM'in çok daha hızlı bir alternatifidir.

| | DDPM | DDIM |
|--|------|------|
| Adım sayısı | T=4000 | 50-500 (seçilebilir) |
| Stokastiklik | Yüksek (gürültü eklenir) | Kontrol edilebilir (η parametresi) |
| Kalite | İyi | Benzer veya daha iyi |

`η=0`: Tamamen deterministik (aynı latent → aynı görüntü)  
`η=1`: DDPM benzeri stokastik


In [ ]:
def generate_ddim(model, batch_size=32, num_steps=50, eta=0.85):
    """
    DDIM (Denoising Diffusion Implicit Models) ile hızlı görüntü üretimi.
    
    DDPM'den farklı olarak, tüm T adımı yerine daha az adım kullanır.
    eta (η) parametresi stokastisite düzeyini kontrol eder:
    - η=0: Tamamen deterministik (aynı gürültü → aynı görüntü)
    - η=1: DDPM benzeri stokastik
    
    Args:
        model: Eğitilmiş diffusion modeli
        batch_size: Üretilecek görüntü sayısı
        num_steps: Kullanılacak adım sayısı (DDPM'den çok daha az olabilir)
        eta: Stokastisite katsayısı [0, 1]
    """
    model.eval()
    with torch.no_grad():
        xt = torch.randn([batch_size, 1, 28, 28], device=device)
        
        # T'den 0'a num_steps adet eşit aralıklı zaman noktası seç
        times = torch.linspace(T - 1, 0, steps=num_steps + 1).long().tolist()
        
        # Ardışık (t, t_prev) çiftleri üzerinde döngü
        for t, t_prev in zip(times[:-1], times[1:]):
            print(f"\rt = {t}", end=" ")
            
            t_batch  = torch.full((batch_size, 1), t, device=device)
            sample   = DiffusionSample(xt, t_batch)
            eps_pred = model(sample)  # ε̂: tahmin edilen gürültü
            
            # x̂₀: Mevcut x_t'den orijinal görüntüyü tahmin et
            x0 = ((xt - (1 - alpha_bars[t]).sqrt() * eps_pred)
                  / (alpha_bars[t].sqrt()))
            
            # DDIM varyans hesabı
            abar_t_prev = alpha_bars[t_prev]
            variance    = eta * (1 - abar_t_prev) / (1 - alpha_bars[t]) * betas[t]
            sigma_t     = variance.sqrt()
            
            # Gürültüsüz yön: tahmin edilen x₀ ile ε̂'dan hesaplanır
            pred_dir = (1 - abar_t_prev - sigma_t**2).sqrt() * eps_pred
            
            noise = torch.randn_like(xt)
            # DDIM güncelleme adımı:
            # x_{t_prev} = √ᾱ_{t_prev} * x̂₀ + pred_dir + σ_t * z
            xt = abar_t_prev.sqrt() * x0 + pred_dir + sigma_t * noise
        
        # [-1,1] → [0,1]
        return torch.clamp((xt + 1) / 2, 0, 1)

torch.manual_seed(42)
X_gen_ddim = generate_ddim(diffusion_model, num_steps=500)
# num_steps=500: DDPM'in 4000 adımına karşılık 8x daha hızlı!

In [ ]:
# DDIM ile üretilen görüntüleri göster
plot_multiple_images(X_gen_ddim, 8)
plt.show()

# 12. Stable Diffusion (Hugging Face `diffusers`)

**Stable Diffusion**, latent uzayda çalışan bir diffusion modeli. Ham piksel uzayı yerine daha küçük bir VAE latent uzayında diffusion yaparak çok daha verimli çalışır.

`sd-turbo`: Stability AI'nin hızlandırılmış, tek adımda görüntü üretebilen modeli.  
Hugging Face `diffusers` kütüphanesi üzerinden kullanılır.

⚠️ Bu hücreyi çalıştırmak için internet bağlantısı ve GPU önerilir.


In [ ]:
from diffusers import AutoPipelineForText2Image

# sd-turbo modelini Hugging Face Hub'dan indir ve yükle
# variant='fp16': 16-bit kayan nokta (daha az VRAM)
# dtype=torch.float16: FP16 ile bellek tasarrufu
pipe = AutoPipelineForText2Image.from_pretrained(
    "stabilityai/sd-turbo",
    variant="fp16",
    dtype=torch.float16
)
pipe.to(device)  # Modeli GPU/CPU'ya taşı

prompt = "A closeup photo of an orangutan reading a book"  # Metin istemi

In [ ]:
torch.manual_seed(42)

# Metin isteminden görüntü üret
# num_inference_steps=1: sd-turbo TEK ADIMDA görüntü üretebilir!
# guidance_scale=0.0: Classifier-free guidance yok (sd-turbo varsayılanı)
image = pipe(prompt=prompt, num_inference_steps=1, guidance_scale=0.0).images[0]

# Görüntüyü kaydet
image.save("my_orangutan_reading.png")

# Görüntüyü göster
plt.imshow(image)
plt.axis("off")
plt.show()

---

# Özet

Bu notebook'ta aşağıdaki üretici model mimarilerini inceledik:

## Autoencoder Ailesi

| Model | Temel Özellik | Kullanım |
|-------|--------------|----------|
| **Undercomplete Linear AE** | Doğrusal, boyut indirgeme | PCA benzeri |
| **Stacked AE** | Derin katmanlar, ReLU/Sigmoid | Genel feature öğrenme |
| **Tied Weights AE** | Ağırlık paylaşımı | Az parametre |
| **Convolutional AE** | Conv/ConvTranspose | Görüntü verisi |
| **Recurrent AE** | LSTM encoder-decoder | Sıralı veri |
| **Denoising AE** | Dropout/GaussianNoise | Robust feature öğrenme |
| **Sparse AE** | KL regularizasyon | Yorumlanabilir coding |

## Üretici Modeller

| Model | Temel Özellik | Güçlü Yönler |
|-------|--------------|-------------|
| **VAE** | Reparameterization trick, KL loss | Sürekli latent space, interpolation |
| **Discrete VAE** | Gumbel-Softmax, kesikli latent | Kategorik temsil |
| **GAN** | Generator + Discriminator | Keskin görüntüler |
| **DCGAN** | CNN tabanlı GAN | Daha iyi görüntü kalitesi |
| **DDPM** | Forward/Reverse diffusion | Çeşitli, yüksek kalite |
| **DDIM** | Hızlı örnekleme | DDPM'den çok hızlı |
| **Stable Diffusion** | Latent diffusion + text | Text-to-image |
